# Beyond the Read — Uncertainty-Aware Graph-Temporal Intelligence for Imperfect RFID Streams

### Reproducible Colab experiment suite (v2 — *pioneering* upgrade)

This notebook implements and evaluates the framework proposed in the manuscript. Relative to a
conventional "simulate → classify → detect" RFID study, this version contributes **six methodological
elements that are, to our knowledge, not jointly available in the RFID analytics literature**:

| # | Contribution | Where |
|---|---|---|
| C1 | **Leakage-free protocol.** Corruption ground truth is *quarantined*: no oracle variable can enter any model input. A machine-checked audit asserts this. | §3 |
| C2 | **Graph-constrained neural CRF** (`GT-CRF`): a Transformer encoder whose *structured* transition potentials are masked by the process graph `G`, trained end-to-end — the direct instantiation of Eq. (2) with `L_obs`, `L_graph`, `L_temp`. | §5 |
| C3 | **Certified minimum-cost counterfactual repair.** Repair is solved *exactly* as edit distance from the decoded route to the **regular language of admissible paths in `G`** (DP over positions × graph nodes). Not a heuristic — the returned edit set is provably minimum cost. | §9 |
| C4 | **Repair-identifiability validation.** Because the simulator retains the corruption mechanism, each counterfactual edit is scored against the *true* injected fault. Counterfactual explanations are therefore *validated*, not merely plausible. | §9 |
| C5 | **Mechanistic sensing-vs-operational diagnosis.** Diagnosis is derived from the *asymmetry of repair cost* — cheap-to-repair-by-observation-edit ⇒ sensing; irreducible route violation ⇒ operational — rather than from a black-box feature classifier. | §10 |
| C6 | **Degradation-adaptive conformal prediction + risk control.** Marginal split conformal *loses coverage* under progressive RFID degradation. We estimate severity `γ̂` from observables only and apply Mondrian γ̂-conditional conformal, restoring coverage; abstention thresholds are then selected with a finite-sample **Learn-then-Test** guarantee that selective risk ≤ ε. | §8, §11 |

> **Scope statement.** The study is simulation-based by design: exact latent ground truth and exact
> corruption provenance are required for C4 and C5, and cannot be obtained from field deployments.
> No claim of physical field validation is made from these experiments alone (see §16).

All artefacts are written to `/content/drive/MyDrive/Outputs/<PROJECT_NAME>/`.

In [ ]:
# ============================================================
# 0. Google Drive mount and project directory layout
# ============================================================
from pathlib import Path
import os, sys, json, math, time, random, warnings, platform, hashlib, itertools
warnings.filterwarnings("ignore")

PROJECT_NAME = "Beyond_the_Read_RFID_AI"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/Outputs") / PROJECT_NAME
else:
    BASE_DIR = Path.cwd() / "Outputs" / PROJECT_NAME

FIG_DIR   = BASE_DIR / "Figures"
TAB_DIR   = BASE_DIR / "Tables"
TEX_DIR   = BASE_DIR / "LaTeX"
MODEL_DIR = BASE_DIR / "Models"
DATA_DIR  = BASE_DIR / "Data"
LOG_DIR   = BASE_DIR / "Logs"
for p in [BASE_DIR, FIG_DIR, TAB_DIR, TEX_DIR, MODEL_DIR, DATA_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project output directory:", BASE_DIR)

## 1. Configuration, determinism and environment capture

`QUICK_RUN=True` gives a fast end-to-end validation pass (a few minutes on a Colab CPU).
Set it to `False` for the final manuscript run. Every reported number is regenerated from
this configuration; the configuration hash is embedded in every exported table.

In [ ]:
# ============================================================
# 1. Global configuration
# ============================================================
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass, asdict, field
from collections import defaultdict, deque

QUICK_RUN = True          # <<< set False for the final manuscript run

CFG = dict(
    project        = PROJECT_NAME,
    seed           = 42,
    n_states       = 8,
    n_assets       = 3000 if QUICK_RUN else 12000,
    n_seeds        = 3     if QUICK_RUN else 10,
    stress_assets  = 700   if QUICK_RUN else 2500,
    stress_levels  = 6     if QUICK_RUN else 9,
    epochs         = 12    if QUICK_RUN else 40,
    batch_size     = 64,
    d_model        = 64,
    n_heads        = 4,
    n_layers       = 2,
    dropout        = 0.1,
    lr             = 2e-3,
    weight_decay   = 1e-4,
    lambda_diag    = 0.5,      # weight of the auxiliary dual-anomaly head
    lambda_graph   = 8.0,      # penalty magnitude for inadmissible transitions (L_graph)
    trans_delta    = 0.25,      # max learned modulation of a graph potential (keeps G structural)
    alpha          = 0.10,     # conformal miscoverage target
    risk_epsilon   = 0.10,     # Learn-then-Test selective-risk budget
    risk_delta     = 0.05,     # Learn-then-Test confidence level
    p_operational  = 0.25,     # prevalence of genuine operational deviations
)

SEED     = CFG["seed"]
N_STATES = CFG["n_states"]
ALPHA    = CFG["alpha"]

def set_all_seeds(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.use_deterministic_algorithms(False)

set_all_seeds(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG_HASH = hashlib.sha256(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:12]
ENV = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
           torch=torch.__version__, device=str(DEVICE), quick_run=QUICK_RUN,
           config_hash=CONFIG_HASH, timestamp=time.strftime("%Y-%m-%d %H:%M:%S"))

with open(BASE_DIR / "config.json", "w") as f:
    json.dump({"config": CFG, "environment": ENV}, f, indent=2)

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "font.size": 10,
    "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False,
})

def save_table(df, name, caption=""):
    '''Persist a result table as CSV *and* as booktabs LaTeX, stamped with the config hash.'''
    df.to_csv(TAB_DIR / f"{name}.csv", index=False)
    try:
        tex = df.to_latex(index=False, escape=True, float_format="%.4f")
    except Exception:
        tex = df.to_string()
    header = f"% {name} | config_hash={CONFIG_HASH} | {ENV['timestamp']}\n"
    if caption:
        header += f"% caption: {caption}\n"
    (TEX_DIR / f"{name}.tex").write_text(header + tex)
    return df

def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")   # vector copy for the manuscript
    return fig

print(json.dumps(ENV, indent=2))

## 2. Process graph `G` and RFID physical abstraction

`G = (V, E)` encodes physically/procedurally admissible transitions between RFID-observable
stages. The canonical route is `S0 → S1 → ... → S7`; two documented shortcuts (`S1→S3`, `S3→S5`)
are *valid but unusual*, which is what makes the sensing-vs-operational distinction non-trivial:
an unusual-but-legal route must **not** be reported as an operational anomaly.

Each stage carries a reader with a stage-dependent RF signature (RSSI mean, phase mean), so the
radio features carry state information that is partially redundant with — and partially
contradictory to — the reported reader identity. That redundancy is exactly what a learned
emission model can exploit when a reader identity is wrong.

In [ ]:
# ============================================================
# 2. Process graph
# ============================================================
STATES = [f"S{i}" for i in range(N_STATES)]
SOURCE, SINK = 0, N_STATES - 1

VALID_EDGES = {(i, i + 1) for i in range(N_STATES - 1)}
VALID_EDGES |= {(1, 3), (3, 5)}                 # documented, legal shortcuts
UNUSUAL_EDGES = {(1, 3), (3, 5)}                # legal but operationally uncommon
SELF_EDGES = {(i, i) for i in range(N_STATES)}  # dwell / repeated reads
VALID_WITH_SELF = VALID_EDGES | SELF_EDGES

ADJ = defaultdict(list)
for a, b in sorted(VALID_EDGES):
    ADJ[a].append(b)

STATE_RSSI_MEAN  = np.linspace(-42.0, -72.0, N_STATES)
STATE_PHASE_MEAN = np.linspace(0.25, 5.75, N_STATES)

def is_valid_transition(a, b, allow_self=True):
    a, b = int(a), int(b)
    return (a, b) in (VALID_WITH_SELF if allow_self else VALID_EDGES)

def graph_inconsistency(seq):
    '''Fraction of inadmissible consecutive transitions in a collapsed route.'''
    seq = [int(s) for s in seq]
    if len(seq) < 2:
        return 0.0
    bad = sum(not is_valid_transition(a, b) for a, b in zip(seq[:-1], seq[1:]))
    return bad / (len(seq) - 1)

def bfs_hops(src, targets_from):
    '''Hop distance from src to every node using the directed admissible edge set.'''
    dist = {src: 0}
    q = deque([src])
    while q:
        u = q.popleft()
        for v in targets_from(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

HOPS_FROM_SOURCE = bfs_hops(SOURCE, lambda u: ADJ[u])
REV_ADJ = defaultdict(list)
for a, b in VALID_EDGES:
    REV_ADJ[b].append(a)
HOPS_TO_SINK = bfs_hops(SINK, lambda u: REV_ADJ[u])

INF = float("inf")
def hops_from_source(v): return HOPS_FROM_SOURCE.get(int(v), INF)
def hops_to_sink(v):     return HOPS_TO_SINK.get(int(v), INF)

def route_is_admissible(route):
    route = [int(s) for s in route]
    if not route or route[0] != SOURCE or route[-1] != SINK:
        return False
    return all((a, b) in VALID_EDGES for a, b in zip(route[:-1], route[1:]))

def collapse(seq):
    '''Collapse consecutive repeats: an observed dwell is not a route change.'''
    out = []
    for s in seq:
        s = int(s)
        if not out or out[-1] != s:
            out.append(s)
    return out

# Transition log-potential used as the graph prior (L_graph of Eq. 2)
def graph_bias_matrix(penalty=None, unusual_penalty=0.7, self_bonus=0.15):
    pen = CFG["lambda_graph"] if penalty is None else penalty
    B = np.full((N_STATES, N_STATES), -pen, dtype=np.float32)
    for i in range(N_STATES):
        B[i, i] = self_bonus
    for a, b in VALID_EDGES:
        B[a, b] = 0.0
    for a, b in UNUSUAL_EDGES:
        B[a, b] = -unusual_penalty
    return B

GRAPH_BIAS = graph_bias_matrix()
print("Admissible edges:", sorted(VALID_EDGES))
print("Hops source->node:", HOPS_FROM_SOURCE)

## 3. The RFID digital environment — and the leakage firewall (**C1**)

Two mechanisms are simulated **independently**, which is what allows the dual anomaly formulation
of Eq. (7) to be evaluated at all:

* **Operational deviation** `A_o` — the *latent* trajectory itself violates the process
  (mandatory-checkpoint bypass, unauthorized jump, reverse movement, excessive dwell).
* **Sensing corruption** `A_s` — the latent trajectory is valid but its RFID *image* is distorted
  (missing reads, duplicates, false reads, RF noise, timestamp delay, reader outage).

`A_s` is deliberately **not** defined as "some corruption operator fired". Under any realistic
nominal setting almost every stream receives some perturbation, which would make the label
degenerate and the reported AUROC meaningless. We define

> `A_s = 1` iff the corruption actually **distorted the observable route**, i.e. the collapsed
> reader sequence differs from the true collapsed route.

### The leakage firewall

The most common — and most damaging — flaw in simulation-based anomaly studies is feeding the
corruption ground truth back to the model as an input feature (`is_duplicate`, `is_false_read`,
`corruption_rate`, ...). A model given those columns is not detecting anything; it is reading the
answer key, and every downstream calibration, conformal and utility number becomes meaningless.

Here every oracle variable is prefixed `oracle_` and lives in a **separate frame**. `§3.3` runs a
machine-checked audit that fails loudly if any oracle column reaches a model input.

In [ ]:
# ============================================================
# 3.1 Corruption configuration and the RFID simulator
# ============================================================
@dataclass
class CorruptionConfig:
    p_miss: float        = 0.10   # missing reads
    p_dup: float         = 0.10   # duplicate reads
    p_false: float       = 0.05   # false / spurious reads at a wrong reader
    rssi_noise: float    = 2.5    # sigma_r
    phase_noise: float   = 0.20   # sigma_phi
    p_delay: float       = 0.08   # timestamp perturbation probability
    delay_scale: float   = 0.9
    p_reader_fail: float = 0.03   # per-reader outage probability

NOMINAL = CorruptionConfig()
CLEAN   = CorruptionConfig(p_miss=0.0, p_dup=0.0, p_false=0.0, rssi_noise=0.6,
                           phase_noise=0.05, p_delay=0.0, p_reader_fail=0.0)

OP_TYPES = ["mandatory_skip", "unauthorized_jump", "reverse", "excess_dwell"]

def sample_latent_route(rng, operational=False):
    '''Sample the *latent* physical route. Returns (route, op_type).'''
    if not operational:
        r = rng.random()
        if r < 0.12:   route = [0, 1, 3, 4, 5, 6, 7]      # legal shortcut
        elif r < 0.20: route = [0, 1, 2, 3, 5, 6, 7]      # legal shortcut
        else:          route = list(range(N_STATES))
        return route, "none"

    op_type = OP_TYPES[rng.integers(len(OP_TYPES))]
    route = list(range(N_STATES))
    if op_type == "mandatory_skip":
        drop = int(rng.integers(2, N_STATES - 1))
        while (drop - 1, drop + 1) in VALID_EDGES:        # ensure the skip is truly illegal
            drop = int(rng.integers(2, N_STATES - 1))
        route = [s for s in route if s != drop]
    elif op_type == "unauthorized_jump":
        a = int(rng.integers(0, N_STATES - 3))
        b = int(rng.integers(a + 2, N_STATES))
        if (a, b) in VALID_EDGES:
            b = min(N_STATES - 1, b + 1)
        route = list(range(a + 1)) + list(range(b, N_STATES))
    elif op_type == "reverse":
        i = int(rng.integers(2, N_STATES - 2))
        route = list(range(i + 1)) + [i - 1] + list(range(i, N_STATES))
    elif op_type == "excess_dwell":
        route = list(range(N_STATES))                     # route legal, dwell pathological
    return route, op_type

def emit_clean_stream(rng, route, op_type):
    '''Expand the route with dwell times and emit one clean RFID event per latent step.'''
    latent, times = [], []
    t = 0.0
    long_dwell_at = int(rng.integers(len(route))) if op_type == "excess_dwell" else -1
    for pos, s in enumerate(route):
        dwell = 1 + int(rng.poisson(0.5))
        if pos == long_dwell_at:
            dwell = int(rng.integers(7, 12))
        dwell = int(np.clip(dwell, 1, 14))
        for _ in range(dwell):
            t += float(rng.exponential(1.0)) + 0.25
            latent.append(int(s)); times.append(t)
    latent = np.asarray(latent, dtype=int)
    times  = np.asarray(times, dtype=float)
    return latent, times

def corrupt_stream(rng, latent, times, cfg):
    '''
    Apply the corruption operator C(X; theta_c).
    Returns an observed event table plus per-event oracle provenance.
    '''
    failed_readers = {s for s in range(N_STATES) if rng.random() < cfg.p_reader_fail}

    obs = []   # dicts: reader, t, rssi, phase, oracle_state, oracle_src, oracle_delayed
    for k, (s, t) in enumerate(zip(latent, times)):
        if s in failed_readers:                      # reader outage: nothing is recorded
            continue
        if rng.random() < cfg.p_miss:                # missed read
            continue
        base = dict(oracle_state=int(s), oracle_latent_idx=int(k))
        obs.append(dict(reader=int(s), t=float(t), oracle_src="clean", **base))
        if rng.random() < cfg.p_dup:                 # duplicate read
            obs.append(dict(reader=int(s), t=float(t) + abs(rng.normal(0.05, 0.03)),
                            oracle_src="duplicate", **base))
        if rng.random() < cfg.p_false:               # spurious detection at a wrong reader
            wrong = int(rng.integers(N_STATES))
            while wrong == int(s):
                wrong = int(rng.integers(N_STATES))
            obs.append(dict(reader=wrong, t=float(t) + abs(rng.normal(0.10, 0.05)),
                            oracle_src="false", **base))

    if not obs:   # degenerate outage: keep one uninformative read so the sequence is not empty
        obs = [dict(reader=int(latent[0]), t=float(times[0]), oracle_src="clean",
                    oracle_state=int(latent[0]), oracle_latent_idx=0)]

    for e in obs:
        true_reader = e["oracle_state"] if e["oracle_src"] != "false" else e["reader"]
        e["rssi"]  = float(STATE_RSSI_MEAN[true_reader] + rng.normal(0, cfg.rssi_noise))
        e["phase"] = float((STATE_PHASE_MEAN[true_reader] + rng.normal(0, cfg.phase_noise)) % (2*np.pi))
        if rng.random() < cfg.p_delay:               # timestamp perturbation / delayed report
            e["t"] += float(rng.exponential(cfg.delay_scale))
            e["oracle_delayed"] = 1
        else:
            e["oracle_delayed"] = 0

    obs.sort(key=lambda e: e["t"])                   # readers only see arrival order
    return obs, failed_readers

def simulate_dataset(n_assets, seed, cfg=NOMINAL, p_operational=None):
    '''Generate a full corpus. Observable and oracle columns are returned in ONE frame but are
       namespaced (`oracle_*`) and separated by `split_observable_oracle` before modelling.'''
    p_op = CFG["p_operational"] if p_operational is None else p_operational
    rng = np.random.default_rng(seed)
    ev_rows, tr_rows = [], []

    for aid in range(n_assets):
        operational = bool(rng.random() < p_op)
        route, op_type = sample_latent_route(rng, operational)
        latent, times = emit_clean_stream(rng, route, op_type)
        obs, failed = corrupt_stream(rng, latent, times, cfg)

        reader_route = collapse([e["reader"] for e in obs])
        true_route   = collapse(latent.tolist())
        A_o = int(op_type != "none")
        A_s = int(reader_route != true_route)        # observation-level distortion (see markdown)

        for j, e in enumerate(obs):
            ev_rows.append(dict(asset_id=aid, event_pos=j, reader=e["reader"], t=e["t"],
                                rssi=e["rssi"], phase=e["phase"],
                                oracle_state=e["oracle_state"], oracle_src=e["oracle_src"],
                                oracle_delayed=e["oracle_delayed"]))
        tr_rows.append(dict(asset_id=aid, op_type=op_type, A_o=A_o, A_s=A_s,
                            n_obs=len(obs), n_latent=int(len(latent)),
                            oracle_true_route=true_route, oracle_latent=latent.tolist(),
                            oracle_failed_readers=sorted(failed),
                            oracle_dwell_max=int(pd.Series(latent).value_counts().max())))

    return pd.DataFrame(ev_rows), pd.DataFrame(tr_rows)

t0 = time.time()
events, trajectories = simulate_dataset(CFG["n_assets"], SEED, NOMINAL)
print(f"Simulated {len(trajectories)} assets / {len(events)} events in {time.time()-t0:.1f}s")
print(trajectories[["A_s", "A_o"]].mean().rename("prevalence").to_frame().T)
print(trajectories["op_type"].value_counts(normalize=True).round(3).to_dict())

In [ ]:
# ============================================================
# 3.2 Observational feature construction  (NO oracle variable is used)
# ============================================================
OBSERVABLE_EVENT_FEATURES = [
    "rssi", "phase_sin", "phase_cos",
    "dt_prev", "dt_next", "dt_prev_z", "t_frac", "pos_frac",
    "same_reader_prev", "same_reader_next", "reader_run_len",
    "reader_share", "rssi_resid", "reader_jump", "n_obs_norm",
]
ORACLE_PREFIX = "oracle_"

def build_event_features(ev):
    '''
    Derive per-event features from *observable* quantities only:
    reader identity, timestamp, RSSI, phase and their within-sequence context.
    `rssi_resid` is the RSSI residual w.r.t. the nominal signature of the *reported* reader —
    a large residual is observable evidence that the reported identity may be wrong.
    '''
    ev = ev.sort_values(["asset_id", "t"]).copy()
    g = ev.groupby("asset_id", sort=False)

    ev["phase_sin"] = np.sin(ev["phase"])
    ev["phase_cos"] = np.cos(ev["phase"])
    ev["dt_prev"]   = g["t"].diff().fillna(0.0)
    ev["dt_next"]   = (-g["t"].diff(-1)).fillna(0.0)

    med = g["dt_prev"].transform("median")
    mad = g["dt_prev"].transform(lambda s: (s - s.median()).abs().median())
    ev["dt_prev_z"] = (ev["dt_prev"] - med) / (mad + 1e-3)

    tmin = g["t"].transform("min"); tmax = g["t"].transform("max")
    ev["t_frac"]   = (ev["t"] - tmin) / (tmax - tmin + 1e-9)
    ev["pos_frac"] = g.cumcount() / (g["t"].transform("size") - 1).clip(lower=1)

    prev_reader = g["reader"].shift(1)
    next_reader = g["reader"].shift(-1)
    ev["same_reader_prev"] = (ev["reader"] == prev_reader).astype(float)
    ev["same_reader_next"] = (ev["reader"] == next_reader).astype(float)
    ev["reader_jump"]      = (ev["reader"] - prev_reader).fillna(0.0)

    # length of the current run of identical reader ids
    newrun = (ev["reader"] != prev_reader) | prev_reader.isna()
    runid = newrun.groupby(ev["asset_id"]).cumsum()
    ev["reader_run_len"] = ev.groupby(["asset_id", runid]).cumcount() + 1

    ev["reader_share"] = ev.groupby(["asset_id", "reader"])["t"].transform("size") / \
                         g["t"].transform("size")
    ev["rssi_resid"]   = ev["rssi"] - STATE_RSSI_MEAN[ev["reader"].to_numpy()]
    ev["n_obs_norm"]   = g["t"].transform("size") / (2.0 * N_STATES)
    ev["event_pos"]    = g.cumcount()
    return ev

events = build_event_features(events)
print("Observable features:", len(OBSERVABLE_EVENT_FEATURES))
display(events.head(4))

In [ ]:
# ============================================================
# 3.3 LEAKAGE AUDIT  (C1) -- fails loudly if any oracle variable can reach a model
# ============================================================
def assert_no_leakage(columns, where=""):
    cols = list(columns)
    bad = [c for c in cols if c.startswith(ORACLE_PREFIX)]
    # names historically used as *inputs* in leaky RFID studies
    forbidden_semantics = ["is_duplicate", "is_false", "is_delayed", "corrupt", "true_state",
                           "true_route", "A_s", "A_o", "op_type", "latent", "failed_reader"]
    bad += [c for c in cols if any(k in c.lower() for k in forbidden_semantics)]
    if bad:
        raise AssertionError(f"LEAKAGE DETECTED in {where}: {sorted(set(bad))}")
    return True

assert_no_leakage(OBSERVABLE_EVENT_FEATURES, "OBSERVABLE_EVENT_FEATURES")
assert_no_leakage(["reader"], "reader identity (categorical input)")

audit = pd.DataFrame([
    {"Group": "Model inputs (observable)", "N": len(OBSERVABLE_EVENT_FEATURES) + 1,
     "Members": ", ".join(["reader"] + OBSERVABLE_EVENT_FEATURES)},
    {"Group": "Quarantined oracle (evaluation only)",
     "N": len([c for c in events.columns if c.startswith(ORACLE_PREFIX)]) +
          len([c for c in trajectories.columns if c.startswith(ORACLE_PREFIX)]) + 3,
     "Members": ", ".join(sorted([c for c in events.columns if c.startswith(ORACLE_PREFIX)] +
                                 [c for c in trajectories.columns if c.startswith(ORACLE_PREFIX)] +
                                 ["A_s", "A_o", "op_type"]))},
])
save_table(audit, "Table_00_leakage_audit",
           "Separation of observable model inputs from quarantined ground truth.")
display(audit)
print("Leakage audit PASSED - no corruption ground truth is visible to any model.")

## 4. Leakage-safe partitioning

Splits are formed at the **asset (trajectory) level**, never at the event level, so no event from a
test trajectory can appear in training. Four partitions are used: `train` (fitting), `val`
(early stopping / model selection), `cal` (**held out for calibration, conformal quantiles and
risk control — never used for fitting**), and `test` (reporting only).

A dedicated calibration split is not optional here: split-conformal coverage and the Learn-then-Test
risk guarantee are only valid on data that was never touched during fitting or selection.

In [ ]:
# ============================================================
# 4. Asset-level train / val / cal / test partition
# ============================================================
def assign_splits(traj_df, seed=SEED, fracs=(0.60, 0.13, 0.13, 0.14)):
    rng = np.random.default_rng(seed)
    aids = traj_df["asset_id"].unique()
    rng.shuffle(aids)
    n = len(aids)
    c = np.cumsum([int(f * n) for f in fracs[:-1]])
    parts = np.split(aids, c)
    mapping = {}
    for name, arr in zip(["train", "val", "cal", "test"], parts):
        for a in arr:
            mapping[a] = name
    return mapping

SPLIT_MAP = assign_splits(trajectories)
trajectories["split"] = trajectories["asset_id"].map(SPLIT_MAP)
events["split"] = events["asset_id"].map(SPLIT_MAP)

split_table = (trajectories.groupby("split")
               .agg(Assets=("asset_id", "size"), Events=("n_obs", "sum"),
                    SensingAnomalyRate=("A_s", "mean"), OperationalAnomalyRate=("A_o", "mean"))
               .reindex(["train", "val", "cal", "test"]).reset_index())
save_table(split_table, "Table_01_partitions", "Asset-level partitioning of the simulated corpus.")
display(split_table)

assert set(trajectories.query("split=='train'").asset_id) & \
       set(trajectories.query("split=='test'").asset_id) == set(), "asset overlap between splits"
print("Partition disjointness verified at the asset level.")

## 5. The proposed model: a graph-constrained neural CRF (**C2**)

The manuscript's estimator (Eq. 2)

$$\widehat{Z}=\arg\min_Z\big[\mathcal{L}_{obs}(X,Z)+\lambda_G\,\mathcal{L}_{graph}(Z;G)+\lambda_T\,\mathcal{L}_{temp}(Z)\big]$$

is instantiated **exactly**, not approximated by a post-hoc filter:

* $\mathcal{L}_{obs}$ — a Transformer encoder $T_\theta$ over the observed event sequence produces
  per-event emission potentials $\psi_t(z_t)$ (Eq. 5).
* $\mathcal{L}_{graph}$ — pairwise transition potentials $A_{ij}$ are **learned but biased by `G`**:
  $A = W + B_G$ where $B_G$ is $0$ on admissible edges, $-\lambda_G$ on inadmissible ones, and mildly
  negative on legal-but-unusual shortcuts. Inadmissible transitions are *penalised, never forbidden*,
  so a genuine operational deviation remains representable and therefore detectable.
* $\mathcal{L}_{temp}$ — temporal irregularity enters through the encoder's continuous time features
  and through the self-transition potential that absorbs dwell and duplicate reads.

The whole thing is trained with the exact **CRF negative log-likelihood** (forward algorithm), so the
graph constraint shapes the *training* objective rather than being bolted on at inference. Decoding
is Viterbi; per-state uncertainty comes from **forward–backward marginals**, which are far better
calibrated than a per-event softmax because they integrate sequence-level evidence.

An auxiliary head on the pooled sequence representation predicts the dual anomaly probabilities
$p(A_s\mid \tilde X,G)$ and $p(A_o\mid \tilde X,G)$ of Eq. (7) jointly with reconstruction —
multi-task learning that lets the diagnosis exploit the reconstruction representation.

**Comparison set.** Note that `RF+Viterbi` (a per-event random forest whose probabilities are
post-hoc Viterbi-decoded) is included as a *baseline*: it is the natural incumbent design, and the
gap between it and `GT-CRF` isolates the value of learning the structure end to end.

In [ ]:
# ============================================================
# 5.1 Tensorisation (padded, masked, asset-level)
# ============================================================
from torch.utils.data import Dataset, DataLoader

FEATS = OBSERVABLE_EVENT_FEATURES

# MAXLEN must accommodate the LONGEST stream we will ever pack, not the 99.5th percentile of the
# nominal corpus. Two reasons: (i) truncating a nominal sequence silently desynchronises it from the
# baseline path arrays; (ii) the stress tests in S12 push duplicate rates far above nominal, making
# observed streams substantially longer -- a percentile-based cap would quietly discard the tail of
# exactly the sequences the robustness study is about. Headroom is sized for the worst compound
# degradation setting.
_obs_len = events.groupby("asset_id").size()
MAXLEN = int(_obs_len.max() * 1.9) + 8
print(f"Observed stream length: median={_obs_len.median():.0f}  max={_obs_len.max()}  "
      f"-> padded length {MAXLEN}")

class FeatureScaler:
    def fit(self, X):
        self.mu = np.nanmean(X, 0); self.sd = np.nanstd(X, 0) + 1e-6; return self
    def transform(self, X):
        return np.clip((X - self.mu) / self.sd, -8, 8).astype(np.float32)

def pack_sequences(ev, tr, scaler=None, fit=False):
    ev = ev.sort_values(["asset_id", "t"])
    tr = tr.set_index("asset_id")
    ids = ev["asset_id"].unique()

    if fit:
        scaler = FeatureScaler().fit(ev[FEATS].to_numpy(np.float64))

    n = len(ids)
    Xn = np.zeros((n, MAXLEN, len(FEATS)), np.float32)
    Xr = np.zeros((n, MAXLEN), np.int64)
    Y  = np.zeros((n, MAXLEN), np.int64)
    M  = np.zeros((n, MAXLEN), bool)
    Ys = np.zeros(n, np.float32); Yo = np.zeros(n, np.float32)
    lens = np.zeros(n, np.int64)
    routes = []

    grouped = dict(list(ev.groupby("asset_id", sort=False)))
    n_trunc = 0
    for i, aid in enumerate(ids):
        g = grouped[aid]
        L = min(len(g), MAXLEN)
        n_trunc += int(len(g) > MAXLEN)
        Xn[i, :L] = scaler.transform(g[FEATS].to_numpy(np.float64)[:L])
        Xr[i, :L] = g["reader"].to_numpy()[:L]
        Y[i, :L]  = g["oracle_state"].to_numpy()[:L]     # supervision target (allowed: it is the label)
        M[i, :L]  = True
        lens[i] = L
        Ys[i] = tr.at[aid, "A_s"]; Yo[i] = tr.at[aid, "A_o"]
        routes.append(list(tr.at[aid, "oracle_true_route"]))

    if n_trunc:
        # Loud, never silent: truncation biases every downstream metric towards short streams.
        print(f"  WARNING: {n_trunc}/{n} sequences exceeded MAXLEN={MAXLEN} and were truncated. "
              f"Increase the MAXLEN headroom before trusting these results.")
    return dict(ids=ids, Xn=Xn, Xr=Xr, Y=Y, M=M, Ys=Ys, Yo=Yo, lens=lens,
                routes=routes), scaler

class SeqDS(Dataset):
    def __init__(self, p): self.p = p
    def __len__(self): return len(self.p["ids"])
    def __getitem__(self, i):
        p = self.p
        return (torch.from_numpy(p["Xn"][i]), torch.from_numpy(p["Xr"][i]),
                torch.from_numpy(p["Y"][i]), torch.from_numpy(p["M"][i]),
                torch.tensor(p["Ys"][i]), torch.tensor(p["Yo"][i]))

PACK, SCALER = {}, None
for name in ["train", "val", "cal", "test"]:
    ev_s = events[events["split"] == name]
    tr_s = trajectories[trajectories["split"] == name]
    PACK[name], SCALER = pack_sequences(ev_s, tr_s, SCALER, fit=(name == "train"))
    print(f"{name:6s} sequences={len(PACK[name]['ids']):5d}  events={int(PACK[name]['lens'].sum())}")

In [ ]:
# ============================================================
# 5.2 Graph-constrained CRF model
# ============================================================
NEG = -1e4

class GraphTemporalCRF(nn.Module):
    '''
    Transformer (or GRU) encoder -> emission potentials; learned transition potentials biased by G.
    use_graph=False  -> ablation with free transitions (no process knowledge)
    encoder='gru'    -> recurrent ablation
    '''
    def __init__(self, n_feat, n_states=N_STATES, d=CFG["d_model"], heads=CFG["n_heads"],
                 layers=CFG["n_layers"], dropout=CFG["dropout"], use_graph=True, encoder="transformer"):
        super().__init__()
        self.n_states, self.use_graph, self.encoder_kind = n_states, use_graph, encoder
        self.reader_emb = nn.Embedding(n_states, d // 2)
        self.proj = nn.Linear(n_feat, d - d // 2)
        self.pos = nn.Parameter(torch.randn(1, MAXLEN, d) * 0.02)
        if encoder == "transformer":
            layer = nn.TransformerEncoderLayer(d_model=d, nhead=heads, dim_feedforward=4*d,
                                               dropout=dropout, batch_first=True, activation="gelu")
            self.enc = nn.TransformerEncoder(layer, num_layers=layers)
        else:
            self.enc = nn.GRU(d, d // 2, num_layers=layers, batch_first=True,
                              bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        self.norm = nn.LayerNorm(d)
        self.emit = nn.Linear(d, n_states)
        # Transitions are parameterised as  A = B_G + delta * tanh(W)  so that the process prior is
        # STRUCTURALLY preserved: learning may modulate an edge potential within +/- delta but can
        # never erase the graph. Without this bound the learned term simply cancels B_G and the
        # process knowledge silently disappears from the objective.
        self.trans = nn.Parameter(torch.zeros(n_states, n_states))
        self.delta = float(CFG.get("trans_delta", 1.0))
        st = torch.full((n_states,), -2.0); st[SOURCE] = 0.0
        en = torch.full((n_states,), -1.0); en[SINK] = 0.0
        self.start = nn.Parameter(st)
        self.end   = nn.Parameter(en)
        self.register_buffer("gbias", torch.tensor(GRAPH_BIAS))
        self.diag_head = nn.Sequential(nn.Linear(2*d, d), nn.GELU(), nn.Dropout(dropout),
                                       nn.Linear(d, 2))     # [A_s logit, A_o logit]

    def transitions(self):
        w = self.delta * torch.tanh(self.trans)
        return self.gbias + w if self.use_graph else self.trans

    def encode(self, xn, xr, mask):
        h = torch.cat([self.proj(xn), self.reader_emb(xr)], -1) + self.pos[:, :xn.size(1)]
        if self.encoder_kind == "transformer":
            h = self.enc(h, src_key_padding_mask=~mask)
        else:
            h, _ = self.enc(h)
        h = self.norm(h)
        em = self.emit(h).masked_fill(~mask.unsqueeze(-1), 0.0)
        hm = h * mask.unsqueeze(-1)
        pooled = torch.cat([hm.sum(1) / mask.sum(1, keepdim=True).clamp(min=1),
                            hm.masked_fill(~mask.unsqueeze(-1), -1e9).max(1).values], -1)
        return em, self.diag_head(pooled)

    # ---- CRF primitives -------------------------------------------------
    def _log_Z(self, em, mask):
        A = self.transitions()
        a = self.start.unsqueeze(0) + em[:, 0]
        for t in range(1, em.size(1)):
            nxt = torch.logsumexp(a.unsqueeze(2) + A.unsqueeze(0), 1) + em[:, t]
            m = mask[:, t].unsqueeze(1)
            a = torch.where(m, nxt, a)
        return torch.logsumexp(a + self.end.unsqueeze(0), 1)

    def _score(self, em, y, mask):
        A = self.transitions()
        B, T = y.shape
        s = self.start[y[:, 0]] + em[:, 0].gather(1, y[:, :1]).squeeze(1)
        for t in range(1, T):
            step = A[y[:, t-1], y[:, t]] + em[:, t].gather(1, y[:, t:t+1]).squeeze(1)
            s = s + step * mask[:, t].float()
        last = mask.sum(1) - 1
        s = s + self.end[y.gather(1, last.unsqueeze(1)).squeeze(1)]
        return s

    def nll(self, em, y, mask):
        return (self._log_Z(em, mask) - self._score(em, y, mask)).mean()

    @torch.no_grad()
    def viterbi(self, em, mask):
        A = self.transitions()
        B, T, S = em.shape
        dp = self.start.unsqueeze(0) + em[:, 0]
        bp = torch.zeros(B, T, S, dtype=torch.long, device=em.device)
        for t in range(1, T):
            sc = dp.unsqueeze(2) + A.unsqueeze(0)
            best, idx = sc.max(1)
            cand = best + em[:, t]
            m = mask[:, t].unsqueeze(1)
            dp = torch.where(m, cand, dp)
            bp[:, t] = idx
        dp = dp + self.end.unsqueeze(0)
        best_last = dp.argmax(1)
        lens = mask.sum(1)
        paths = torch.zeros(B, T, dtype=torch.long, device=em.device)
        for b in range(B):
            L = int(lens[b]); cur = int(best_last[b]); paths[b, L-1] = cur
            for t in range(L-1, 0, -1):
                cur = int(bp[b, t, cur]); paths[b, t-1] = cur
        return paths, dp.max(1).values

    @torch.no_grad()
    def marginals(self, em, mask):
        '''Forward-backward posterior p(z_t | X, G): sequence-aware per-event uncertainty.'''
        A = self.transitions()
        B, T, S = em.shape
        fwd = torch.full((B, T, S), NEG, device=em.device)
        fwd[:, 0] = self.start.unsqueeze(0) + em[:, 0]
        for t in range(1, T):
            nxt = torch.logsumexp(fwd[:, t-1].unsqueeze(2) + A.unsqueeze(0), 1) + em[:, t]
            fwd[:, t] = torch.where(mask[:, t].unsqueeze(1), nxt, fwd[:, t-1])
        bwd = torch.full((B, T, S), NEG, device=em.device)
        lens = mask.sum(1)
        for b in range(B):
            bwd[b, int(lens[b])-1] = self.end
        for t in range(T-2, -1, -1):
            prv = torch.logsumexp(A.unsqueeze(0) + (em[:, t+1] + bwd[:, t+1]).unsqueeze(1), 2)
            upd = mask[:, t+1].unsqueeze(1)
            bwd[:, t] = torch.where(upd, prv, bwd[:, t])
        lp = fwd + bwd
        lp = lp - torch.logsumexp(lp, -1, keepdim=True)
        return lp.exp()

    def path_logprob(self, em, y, mask):
        return self._score(em, y, mask) - self._log_Z(em, mask)

print("GraphTemporalCRF defined.")

In [ ]:
# ============================================================
# 5.3 Training loop
# ============================================================
def levenshtein_lite(a, b):
    a, b = list(a), list(b)
    if not a: return len(b)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def make_loader(pack, shuffle, bs=None):
    return DataLoader(SeqDS(pack), batch_size=bs or CFG["batch_size"], shuffle=shuffle)

def train_model(train_pack, val_pack, use_graph=True, encoder="transformer",
                epochs=None, tag="GT-CRF", verbose=True, seed=SEED):
    set_all_seeds(seed)
    epochs = epochs or CFG["epochs"]
    model = GraphTemporalCRF(len(FEATS), use_graph=use_graph, encoder=encoder).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl, vl = make_loader(train_pack, True), make_loader(val_pack, False)

    best, best_state, hist = -np.inf, None, []
    t0 = time.time()
    for ep in range(epochs):
        model.train(); tot = 0.0
        for xn, xr, y, m, ys, yo in tl:
            xn, xr, y, m = xn.to(DEVICE), xr.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
            ys, yo = ys.to(DEVICE), yo.to(DEVICE)
            em, dlog = model.encode(xn, xr, m)
            loss = model.nll(em, y, m) / m.sum(1).float().mean()
            loss = loss + CFG["lambda_diag"] * (
                F.binary_cross_entropy_with_logits(dlog[:, 0], ys) +
                F.binary_cross_entropy_with_logits(dlog[:, 1], yo))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0); opt.step()
            tot += float(loss)
        sched.step()

        # --- model selection on the metric that actually matters: TRAJECTORY reconstruction ---
        # Per-event accuracy is a poor selection criterion here: a model can score well on it while
        # still producing physically impossible routes. Selection uses validation route similarity.
        model.eval(); corr = n = 0; sims = []; k = 0
        with torch.no_grad():
            for xn, xr, y, m, ys, yo in vl:
                xn, xr, y, m = xn.to(DEVICE), xr.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
                em, _ = model.encode(xn, xr, m)
                path, _ = model.viterbi(em, m)
                corr += int(((path == y) & m).sum()); n += int(m.sum())
                pl = path.cpu().numpy(); ln = m.sum(1).cpu().numpy()
                for b in range(len(pl)):
                    dec = collapse(pl[b, :int(ln[b])])
                    true_route = val_pack["routes"][k]; k += 1
                    d = levenshtein_lite(true_route, dec)
                    sims.append(1.0 - d / max(len(true_route), len(dec), 1))
        acc = corr / max(n, 1); sim = float(np.mean(sims))
        hist.append(dict(epoch=ep, train_loss=tot/max(len(tl),1), val_state_acc=acc,
                         val_route_similarity=sim))
        if sim > best:
            best = sim
            best_state = {k2: v.detach().clone() for k2, v in model.state_dict().items()}
        if verbose and (ep % max(1, epochs // 6) == 0 or ep == epochs - 1):
            print(f"  [{tag}] epoch {ep:3d}  loss={tot/max(len(tl),1):.4f}  "
                  f"val_state_acc={acc:.4f}  val_route_sim={sim:.4f}")

    model.load_state_dict(best_state)
    model.eval()
    return model, pd.DataFrame(hist), time.time() - t0

# ---------------------------------------------------------------------------
# lambda_G is SELECTED ON THE VALIDATION SPLIT, not hand-tuned on test.
# The sweep is reported as a sensitivity analysis so the choice is auditable.
# ---------------------------------------------------------------------------
LAMBDA_GRID = [4.0, 8.0] if QUICK_RUN else [2.0, 4.0, 8.0, 12.0]

def val_route_similarity(model, pack):
    inf = run_inference_lite(model, pack)
    sims = []
    for i in range(len(pack["ids"])):
        dec = collapse(inf[i])
        tr_ = pack["routes"][i]
        sims.append(1.0 - levenshtein_lite(tr_, dec) / max(len(tr_), len(dec), 1))
    return float(np.mean(sims))

@torch.no_grad()
def run_inference_lite(model, pack, batch=128):
    model.eval(); out = []
    for xn, xr, y, m, ys, yo in DataLoader(SeqDS(pack), batch_size=batch, shuffle=False):
        xn, xr, m = xn.to(DEVICE), xr.to(DEVICE), m.to(DEVICE)
        em, _ = model.encode(xn, xr, m)
        path, _ = model.viterbi(em, m)
        pl = path.cpu().numpy(); ln = m.sum(1).cpu().numpy()
        out += [pl[b, :int(ln[b])] for b in range(len(pl))]
    return out

print("Selecting lambda_G on the validation split ...")
sweep_rows, candidates = [], {}
for lg in LAMBDA_GRID:
    CFG["lambda_graph"] = lg
    GRAPH_BIAS = graph_bias_matrix()
    m_, h_, s_ = train_model(PACK["train"], PACK["val"], True, "transformer",
                             tag=f"GT-CRF(lg={lg})", verbose=False)
    vs = val_route_similarity(m_, PACK["val"])
    sweep_rows.append(dict(LambdaGraph=lg, TransDelta=CFG["trans_delta"],
                           ValRouteSimilarity=vs, TrainSeconds=s_))
    candidates[lg] = (m_, h_, s_)
    print(f"  lambda_G={lg:5.1f} -> validation route similarity {vs:.4f}")

sweep_df = pd.DataFrame(sweep_rows)
save_table(sweep_df.round(4), "Table_02b_lambda_graph_sensitivity",
           "Sensitivity to the graph-penalty weight; selection made on validation only.")
LAMBDA_STAR_G = float(sweep_df.loc[sweep_df["ValRouteSimilarity"].idxmax(), "LambdaGraph"])
CFG["lambda_graph"] = LAMBDA_STAR_G
GRAPH_BIAS = graph_bias_matrix()
MODEL, HIST, TRAIN_SECS = candidates[LAMBDA_STAR_G]
print(f"Selected lambda_G = {LAMBDA_STAR_G} (validation-optimal)")
torch.save(MODEL.state_dict(), MODEL_DIR / "gt_crf.pt")
save_table(HIST, "Table_02_training_history", "Training curve of the proposed GT-CRF.")

fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.6))
ax[0].plot(HIST["epoch"], HIST["train_loss"]); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("CRF loss")
ax[0].set_title("Training objective")
ax[1].plot(HIST["epoch"], HIST["val_state_acc"], color="tab:green", label="state accuracy")
ax[1].plot(HIST["epoch"], HIST["val_route_similarity"], color="tab:purple", label="route similarity")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("validation metric"); ax[1].set_title("Validation"); ax[1].legend(fontsize=8)
save_fig(fig, "Fig_01_training"); plt.show()
print(f"Training time: {TRAIN_SECS:.1f}s on {DEVICE}")

In [ ]:
# ============================================================
# 5.4 Inference helper: run a model over a packed split
# ============================================================
@torch.no_grad()
def run_inference(model, pack, batch=128):
    model.eval()
    out = dict(paths=[], marg=[], logp_gold=[], diag=[], lens=[], ids=pack["ids"])
    ds = SeqDS(pack); dl = DataLoader(ds, batch_size=batch, shuffle=False)
    for xn, xr, y, m, ys, yo in dl:
        xn, xr, y, m = xn.to(DEVICE), xr.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
        em, dlog = model.encode(xn, xr, m)
        path, _ = model.viterbi(em, m)
        marg = model.marginals(em, m)
        lp = model.path_logprob(em, y, m)
        out["paths"].append(path.cpu().numpy())
        out["marg"].append(marg.cpu().numpy())
        out["logp_gold"].append(lp.cpu().numpy())
        out["diag"].append(torch.sigmoid(dlog).cpu().numpy())
        out["lens"].append(m.sum(1).cpu().numpy())
    for k in ["paths", "marg", "logp_gold", "diag", "lens"]:
        out[k] = np.concatenate(out[k], 0)
    return out

INF_TEST = run_inference(MODEL, PACK["test"])
INF_CAL  = run_inference(MODEL, PACK["cal"])
INF_VAL  = run_inference(MODEL, PACK["val"])
print("Inference complete.  test sequences:", len(INF_TEST["ids"]))

## 6. Baselines and trajectory reconstruction

Five comparators span the space the manuscript promises: a rule, a classical per-event learner, the
incumbent "learn emissions then Viterbi" design, and two neural sequence models without process
knowledge.

| Method | Emissions | Temporal | Graph `G` |
|---|---|---|---|
| `RawReader` | reported reader id | – | – |
| `LogReg` | logistic regression per event | – | – |
| `RF` | random forest per event | – | – |
| `RF+Viterbi` | random forest per event | post-hoc Viterbi | inference only |
| `GRU-CRF (no G)` | BiGRU | CRF | – |
| `Transformer-CRF (no G)` | Transformer | CRF | – |
| **`GT-CRF` (proposed)** | Transformer | CRF | **in the training objective** |

Reconstruction quality is reported as per-event latent-state accuracy and macro-F1, plus
**trajectory** measures on the collapsed route: normalised Levenshtein similarity, exact-route
recovery rate, and residual graph inconsistency (how often the reconstructed route still contains a
physically impossible transition).

Two cautions on reading this table. First, **per-event accuracy is the least important column**: a
model can score well on it while emitting routes that are physically impossible, which is precisely
the failure the framework exists to prevent. Exact-route recovery and graph inconsistency are the
operationally meaningful measures. Second, **rank order in a single run is not evidence** — §13.2
repeats every comparison across seeds and applies a Holm-corrected paired test with effect sizes.
Where that test does not separate two methods, the manuscript must report them as tied.

In [ ]:
# ============================================================
# 6.1 Non-neural baselines
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

def flat_xy(split):
    ev = events[events["split"] == split].sort_values(["asset_id", "t"])
    X = ev[FEATS].to_numpy(np.float64)
    Xr = pd.get_dummies(pd.Categorical(ev["reader"], categories=range(N_STATES))).to_numpy(float)
    return np.hstack([X, Xr]), ev["oracle_state"].to_numpy(), ev

Xtr, ytr, _ = flat_xy("train")
Xte, yte, ev_te = flat_xy("test")
Xcal, ycal, ev_cal = flat_xy("cal")
assert_no_leakage(FEATS, "flat baseline design matrix")

t0 = time.time()
RF = RandomForestClassifier(n_estimators=200 if QUICK_RUN else 500, min_samples_leaf=2,
                            n_jobs=-1, random_state=SEED).fit(Xtr, ytr)
RF_SECS = time.time() - t0
LR = Pipeline([("s", StandardScaler()), ("c", LogisticRegression(max_iter=600, n_jobs=-1))]).fit(Xtr, ytr)
print(f"RF fitted in {RF_SECS:.1f}s")

def viterbi_numpy(logem, trans_log):
    n, S = logem.shape
    dp = logem[0].copy(); bp = np.zeros((n, S), int)
    prior = np.full(S, -2.0); prior[SOURCE] = 0.0
    dp = dp + prior
    for t in range(1, n):
        sc = dp[:, None] + trans_log
        bp[t] = sc.argmax(0); dp = sc.max(0) + logem[t]
    path = np.zeros(n, int); path[-1] = dp.argmax()
    for t in range(n-1, 0, -1):
        path[t-1] = bp[t, path[t]]
    return path

TRANS_LOG_NP = GRAPH_BIAS.astype(np.float64)

def rf_viterbi_paths(ev_split, model=RF):
    Xs, _, ev = flat_xy(ev_split)
    proba = model.predict_proba(Xs)
    logem = np.log(np.clip(proba, 1e-12, 1))
    out = {}
    idx = 0
    for aid, g in ev.groupby("asset_id", sort=False):
        L = len(g)
        out[aid] = viterbi_numpy(logem[idx:idx+L], TRANS_LOG_NP)
        idx += L
    return out, proba, ev

RFV_PATHS, RF_PROBA_TEST, _ = rf_viterbi_paths("test")
print("RF+Viterbi decoded", len(RFV_PATHS), "test trajectories")

In [ ]:
# ============================================================
# 6.2 Neural ablation baselines (no process graph)
# ============================================================
print("Training GRU-CRF (no G) ...")
M_GRU, H_GRU, S_GRU = train_model(PACK["train"], PACK["val"], use_graph=False,
                                  encoder="gru", tag="GRU-CRF(noG)")
print("Training Transformer-CRF (no G) ...")
M_TNG, H_TNG, S_TNG = train_model(PACK["train"], PACK["val"], use_graph=False,
                                  encoder="transformer", tag="TF-CRF(noG)")
INF_GRU = run_inference(M_GRU, PACK["test"])
INF_TNG = run_inference(M_TNG, PACK["test"])

In [ ]:
# ============================================================
# 6.3 Reconstruction metrics
# ============================================================
def levenshtein(a, b):
    a, b = list(a), list(b)
    if not a: return len(b)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def route_similarity(true_route, pred_route):
    d = levenshtein(true_route, pred_route)
    return 1.0 - d / max(len(true_route), len(pred_route), 1)

TRAJ_IDX = trajectories.set_index("asset_id")

def evaluate_paths(ids, paths, lens, name):
    '''paths: (N, MAXLEN) int; lens: (N,) — evaluate event- and trajectory-level reconstruction.'''
    ev_acc_num = ev_acc_den = 0
    y_all, p_all = [], []
    sims, exact, ginc = [], [], []
    for i, aid in enumerate(ids):
        L = int(lens[i])
        p = np.asarray(paths[i][:L], int)
        y = PACK_LOOKUP[aid]["y"][:L]
        y_all.append(y); p_all.append(p)
        ev_acc_num += int((p == y).sum()); ev_acc_den += L
        tr_route = TRAJ_IDX.at[aid, "oracle_true_route"]
        pr_route = collapse(p)
        sims.append(route_similarity(tr_route, pr_route))
        exact.append(float(list(tr_route) == list(pr_route)))
        ginc.append(graph_inconsistency(pr_route))
    y_all = np.concatenate(y_all); p_all = np.concatenate(p_all)
    return dict(Method=name,
                StateAccuracy=ev_acc_num / ev_acc_den,
                StateMacroF1=f1_score(y_all, p_all, average="macro"),
                TrajSimilarity=float(np.mean(sims)),
                ExactRouteRate=float(np.mean(exact)),
                GraphInconsistency=float(np.mean(ginc)))

# fast lookup of per-asset ground truth aligned with the packed order
PACK_LOOKUP = {}
for split in ["test", "cal", "val"]:
    p = PACK[split]
    for i, aid in enumerate(p["ids"]):
        PACK_LOOKUP[aid] = dict(y=p["Y"][i], L=int(p["lens"][i]))

def paths_from_dict(ids, d, lens):
    '''Re-index a per-asset path dictionary into the packed (N, MAXLEN) layout.
       Baseline paths are cut to the packed length so every method is scored on exactly the
       same events -- a mismatch here would compare methods over different event sets.'''
    out = np.zeros((len(ids), MAXLEN), int)
    for i, aid in enumerate(ids):
        L = min(len(d[aid]), int(lens[i]), MAXLEN)
        out[i, :L] = np.asarray(d[aid])[:L]
    return out

ids_te, lens_te = PACK["test"]["ids"], PACK["test"]["lens"]

# raw reader / per-event learners, re-ordered into packed order
raw_paths, lr_paths, rf_paths = {}, {}, {}
lr_pred = LR.predict(Xte); rf_pred = RF.predict(Xte)
idx = 0
for aid, g in ev_te.groupby("asset_id", sort=False):
    L = len(g)
    raw_paths[aid] = g["reader"].to_numpy()
    lr_paths[aid]  = lr_pred[idx:idx+L]
    rf_paths[aid]  = rf_pred[idx:idx+L]
    idx += L

recon_rows = [
    evaluate_paths(ids_te, paths_from_dict(ids_te, raw_paths, lens_te), lens_te, "RawReader"),
    evaluate_paths(ids_te, paths_from_dict(ids_te, lr_paths, lens_te), lens_te, "LogReg"),
    evaluate_paths(ids_te, paths_from_dict(ids_te, rf_paths, lens_te), lens_te, "RF"),
    evaluate_paths(ids_te, paths_from_dict(ids_te, RFV_PATHS, lens_te), lens_te, "RF+Viterbi"),
    evaluate_paths(ids_te, INF_GRU["paths"], lens_te, "GRU-CRF (no G)"),
    evaluate_paths(ids_te, INF_TNG["paths"], lens_te, "Transformer-CRF (no G)"),
    evaluate_paths(ids_te, INF_TEST["paths"], lens_te, "GT-CRF (proposed)"),
]
recon = pd.DataFrame(recon_rows)
save_table(recon.round(4), "Table_03_reconstruction",
           "Latent-state and trajectory reconstruction on the held-out test partition.")
display(recon.round(4))

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for k, (col, title) in enumerate([("TrajSimilarity", "Trajectory similarity"),
                                  ("ExactRouteRate", "Exact route recovery"),
                                  ("GraphInconsistency", "Residual graph inconsistency")]):
    d = recon.sort_values(col, ascending=(col == "GraphInconsistency"))
    colors = ["tab:red" if m.startswith("GT-CRF") else "tab:blue" for m in d["Method"]]
    ax[k].barh(d["Method"], d[col], color=colors)
    ax[k].set_title(title); ax[k].set_xlabel(col)
save_fig(fig, "Fig_02_reconstruction"); plt.show()

## 7. Calibration of the reconstruction posterior

The CRF forward–backward marginals give $p(z_t\mid \tilde X,G)$. We report Expected Calibration
Error, multiclass Brier score and reliability curves before and after **temperature scaling fitted on
the calibration partition only**. Calibration is a prerequisite for everything in §8–§11: an
uncalibrated posterior makes conformal sets, abstention thresholds and the utility model unreliable.

In [ ]:
# ============================================================
# 7. Calibration
# ============================================================
def flatten_marginals(inf, pack):
    P, Y = [], []
    for i in range(len(inf["ids"])):
        L = int(inf["lens"][i])
        P.append(inf["marg"][i, :L]); Y.append(pack["Y"][i, :L])
    return np.concatenate(P, 0), np.concatenate(Y, 0)

P_cal, Y_cal = flatten_marginals(INF_CAL, PACK["cal"])
P_te,  Y_te  = flatten_marginals(INF_TEST, PACK["test"])

def ece(y, p, bins=15):
    conf = p.max(1); pred = p.argmax(1); corr = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum(): e += m.mean() * abs(corr[m].mean() - conf[m].mean())
    return float(e)

def brier(y, p):
    Y1 = np.eye(p.shape[1])[y]
    return float(((p - Y1) ** 2).sum(1).mean())

def nll(y, p):
    return float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean())

def fit_temperature(p, y, grid=np.linspace(0.4, 3.0, 53)):
    logp = np.log(np.clip(p, 1e-12, 1))
    best, bestT = np.inf, 1.0
    for T in grid:
        q = np.exp(logp / T); q /= q.sum(1, keepdims=True)
        v = nll(y, q)
        if v < best: best, bestT = v, T
    return float(bestT)

TEMP = fit_temperature(P_cal, Y_cal)
def apply_T(p, T=None):
    T = TEMP if T is None else T
    q = np.exp(np.log(np.clip(p, 1e-12, 1)) / T)
    return q / q.sum(1, keepdims=True)

P_te_cal = apply_T(P_te)
calib = pd.DataFrame([
    dict(Posterior="CRF marginals (raw)", Temperature=1.0, ECE=ece(Y_te, P_te),
         Brier=brier(Y_te, P_te), NLL=nll(Y_te, P_te)),
    dict(Posterior="CRF marginals (temperature-scaled)", Temperature=TEMP, ECE=ece(Y_te, P_te_cal),
         Brier=brier(Y_te, P_te_cal), NLL=nll(Y_te, P_te_cal)),
    dict(Posterior="RF per-event softmax", Temperature=1.0, ECE=ece(yte, RF_PROBA_TEST),
         Brier=brier(yte, RF_PROBA_TEST), NLL=nll(yte, RF_PROBA_TEST)),
]).round(4)
save_table(calib, "Table_04_calibration", "Calibration of the reconstruction posterior.")
display(calib)

def reliability(y, p, bins=12):
    conf = p.max(1); corr = (p.argmax(1) == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 20: xs.append(conf[m].mean()); ys.append(corr[m].mean())
    return xs, ys

fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
for lbl, p, yy in [("CRF marginals", P_te, Y_te), ("CRF + temperature", P_te_cal, Y_te),
                   ("RF per-event", RF_PROBA_TEST, yte)]:
    xs, ys = reliability(yy, p); ax.plot(xs, ys, marker="o", ms=4, label=lbl)
ax.set_xlabel("mean predicted confidence"); ax.set_ylabel("empirical accuracy")
ax.set_title("Reliability of latent-state posteriors"); ax.legend()
save_fig(fig, "Fig_03_reliability"); plt.show()

## 8. Conformal prediction — and why the standard recipe fails on degrading RFID (**C6**)

### 8.1 Two levels of guarantee

* **Event level (APS).** Adaptive Prediction Sets on the calibrated marginals give a set
  $\Gamma_\alpha(x_t)\subseteq V$ per event with marginal coverage $\ge 1-\alpha$.
* **Trajectory level.** A *sequence-valid* guarantee is the operationally meaningful one — a
  warehouse cares whether the whole reconstructed route is trustworthy, not whether each read is.
  We use the normalised CRF path log-probability as the nonconformity score
  $s(Z)=-\tfrac{1}{T}\log p(Z\mid \tilde X,G)$, calibrate $\hat q$ on the calibration split, and
  report coverage of the true trajectory together with a beam-enumerated lower bound on set size.

### 8.2 The failure mode this notebook exposes

Split conformal guarantees coverage **only under exchangeability**. Progressive RFID degradation is
precisely a violation of it: the test distribution drifts away from the calibration distribution as
readers fail. We show that marginal conformal coverage **collapses below the nominal level** as
severity $\gamma$ increases — a result that, to our knowledge, has not been reported for RFID
analytics, and which invalidates naive "we used conformal prediction, so we have a guarantee" claims.

### 8.3 The fix: degradation-adaptive (Mondrian) conformal

A severity proxy $\hat\gamma$ is regressed **from observables only** (reader-repeat rate, timing
dispersion, RSSI residual spread, raw-sequence impossible-transition rate, read-count deficit).
Calibration examples are then binned by $\hat\gamma$ and a *separate* quantile is used per bin.
Because $\hat\gamma$ uses no ground truth, this is deployable: coverage is restored across the whole
degradation range at the cost of larger sets where the evidence is genuinely poor — which is the
honest behaviour.

In [ ]:
# ============================================================
# 8.1 Event-level adaptive prediction sets (APS)
# ============================================================
def aps_scores(p, y):
    '''APS nonconformity: cumulative probability mass required to reach the true label.'''
    order = np.argsort(-p, axis=1)
    sorted_p = np.take_along_axis(p, order, 1)
    cum = np.cumsum(sorted_p, 1)
    rank = np.argmax(order == y[:, None], axis=1)
    return cum[np.arange(len(y)), rank]

def conformal_quantile(scores, alpha=ALPHA):
    n = len(scores)
    if n == 0: return 1.0
    q = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(scores, q, method="higher"))

def aps_sets(p, qhat):
    order = np.argsort(-p, axis=1)
    sorted_p = np.take_along_axis(p, order, 1)
    cum = np.cumsum(sorted_p, 1)
    keep = cum - sorted_p < qhat          # include labels until the mass threshold is crossed
    sets = np.zeros_like(p, bool)
    np.put_along_axis(sets, order, keep, 1)
    sets[np.arange(len(p)), p.argmax(1)] = True
    return sets

P_cal_T = apply_T(P_cal)
s_cal = aps_scores(P_cal_T, Y_cal)
QHAT_EVENT = conformal_quantile(s_cal, ALPHA)
sets_te = aps_sets(P_te_cal, QHAT_EVENT)

event_cov = float(sets_te[np.arange(len(Y_te)), Y_te].mean())
event_size = float(sets_te.sum(1).mean())
print(f"Event-level APS: coverage={event_cov:.4f} (target {1-ALPHA:.2f}), mean |set|={event_size:.3f}")

In [ ]:
# ============================================================
# 8.2 Trajectory-level conformal sets over admissible paths
# ============================================================
@torch.no_grad()
def beam_paths(model, em, mask, K=24):
    '''Top-K sequence beam under the CRF potentials; returns paths and their log-probabilities.'''
    A = model.transitions()
    B, T, S = em.shape
    logZ = model._log_Z(em, mask)
    beams = [[(float(model.start[s] + em[b, 0, s]), [s]) for s in range(S)] for b in range(B)]
    beams = [sorted(bm, key=lambda z: -z[0])[:K] for bm in beams]
    lens = mask.sum(1)
    for t in range(1, T):
        for b in range(B):
            if t >= int(lens[b]):
                continue
            cand = []
            for sc, pth in beams[b]:
                last = pth[-1]
                for s in range(S):
                    cand.append((sc + float(A[last, s] + em[b, t, s]), pth + [s]))
            cand.sort(key=lambda z: -z[0])
            beams[b] = cand[:K]
    out = []
    for b in range(B):
        finished = [(sc + float(model.end[pth[-1]]) - float(logZ[b]), pth) for sc, pth in beams[b]]
        finished.sort(key=lambda z: -z[0])
        out.append(finished)
    return out

def traj_nonconformity(logp_gold, lens):
    '''s(Z) = -(1/T) log p(Z | X, G): length-normalised so short and long streams are comparable.'''
    return -np.asarray(logp_gold) / np.maximum(np.asarray(lens), 1)

s_traj_cal = traj_nonconformity(INF_CAL["logp_gold"], INF_CAL["lens"])
QHAT_TRAJ = conformal_quantile(s_traj_cal, ALPHA)
s_traj_te = traj_nonconformity(INF_TEST["logp_gold"], INF_TEST["lens"])
traj_cov = float((s_traj_te <= QHAT_TRAJ).mean())

# beam-enumerated LOWER BOUND on the trajectory set size (exact counting is intractable)
set_sizes = []
sub = SeqDS(PACK["test"])
dl = DataLoader(sub, batch_size=32, shuffle=False)
BEAM_LIMIT = 12 if QUICK_RUN else 40
seen_batches = 0
with torch.no_grad():
    for xn, xr, y, m, ys, yo in dl:
        if seen_batches >= BEAM_LIMIT: break
        xn, xr, m = xn.to(DEVICE), xr.to(DEVICE), m.to(DEVICE)
        em, _ = MODEL.encode(xn, xr, m)
        beams = beam_paths(MODEL, em, m, K=24)
        L = m.sum(1).cpu().numpy()
        for b, fin in enumerate(beams):
            sizes = sum(1 for lp, _ in fin if (-lp / max(L[b], 1)) <= QHAT_TRAJ)
            set_sizes.append(sizes)
        seen_batches += 1

conf_tab = pd.DataFrame([
    dict(Level="Event (APS)", Alpha=ALPHA, TargetCoverage=1-ALPHA, EmpiricalCoverage=event_cov,
         MeanSetSize=event_size, SingletonRate=float((sets_te.sum(1) == 1).mean())),
    dict(Level="Trajectory (CRF path score)", Alpha=ALPHA, TargetCoverage=1-ALPHA,
         EmpiricalCoverage=traj_cov, MeanSetSize=float(np.mean(set_sizes)) if set_sizes else np.nan,
         SingletonRate=float(np.mean([s == 1 for s in set_sizes])) if set_sizes else np.nan),
]).round(4)
save_table(conf_tab, "Table_05_conformal",
           "Split-conformal coverage at the event and trajectory level (in-distribution).")
display(conf_tab)
print("Note: trajectory set size is a beam-K lower bound, reported as such in the manuscript.")

In [ ]:
# ============================================================
# 8.3 Observable severity estimator  gamma_hat   (uses NO ground truth)
# ============================================================
SEVERITY_FEATURES = ["repeat_rate", "impossible_rate", "dt_cv", "rssi_resid_sd",
                     "read_deficit", "reader_entropy", "backstep_rate"]

def severity_features(ev):
    rows = []
    for aid, g in ev.sort_values(["asset_id", "t"]).groupby("asset_id", sort=False):
        r = g["reader"].to_numpy()
        t = g["t"].to_numpy()
        dt = np.diff(t) if len(t) > 1 else np.array([0.0])
        col = collapse(r)
        trans = list(zip(col[:-1], col[1:]))
        imposs = np.mean([0.0 if (a, b) in VALID_EDGES else 1.0 for a, b in trans]) if trans else 0.0
        back = np.mean([1.0 if b < a else 0.0 for a, b in trans]) if trans else 0.0
        _, cnt = np.unique(r, return_counts=True)
        pr = cnt / cnt.sum()
        rows.append(dict(asset_id=aid,
                         repeat_rate=float(np.mean(r[1:] == r[:-1])) if len(r) > 1 else 0.0,
                         impossible_rate=float(imposs),
                         dt_cv=float(dt.std() / (dt.mean() + 1e-6)),
                         rssi_resid_sd=float((g["rssi"].to_numpy() -
                                              STATE_RSSI_MEAN[r]).std()),
                         read_deficit=float(max(0.0, 1.0 - len(r) / (1.6 * N_STATES))),
                         reader_entropy=float(-(pr * np.log(pr + 1e-12)).sum()),
                         backstep_rate=float(back)))
    return pd.DataFrame(rows)

def compound_config(gamma):
    return CorruptionConfig(p_miss=0.02 + 0.45*gamma, p_dup=0.02 + 0.33*gamma,
                            p_false=0.01 + 0.22*gamma, rssi_noise=1.0 + 8.0*gamma,
                            phase_noise=0.06 + 0.60*gamma, p_delay=0.02 + 0.30*gamma,
                            p_reader_fail=0.00 + 0.28*gamma)

print("Fitting the observable severity estimator gamma_hat ...")
from sklearn.ensemble import GradientBoostingRegressor
sev_X, sev_y = [], []
for gam in np.linspace(0, 1, 7):
    ev_g, tr_g = simulate_dataset(400 if QUICK_RUN else 900, seed=SEED + 900 + int(gam * 97),
                                  cfg=compound_config(gam))
    sf = severity_features(ev_g)
    sev_X.append(sf[SEVERITY_FEATURES].to_numpy()); sev_y.append(np.full(len(sf), gam))
sev_X = np.vstack(sev_X); sev_y = np.concatenate(sev_y)
GAMMA_MODEL = GradientBoostingRegressor(random_state=SEED, n_estimators=200, max_depth=3).fit(sev_X, sev_y)
print("gamma_hat train R^2:", round(GAMMA_MODEL.score(sev_X, sev_y), 3))
assert_no_leakage(SEVERITY_FEATURES, "severity estimator inputs")

def estimate_gamma(ev):
    sf = severity_features(ev)
    sf["gamma_hat"] = np.clip(GAMMA_MODEL.predict(sf[SEVERITY_FEATURES].to_numpy()), 0, 1)
    return sf[["asset_id", "gamma_hat"]]

In [ ]:
# ============================================================
# 8.4 Coverage under progressive degradation: marginal vs Mondrian gamma-conditional conformal
# ============================================================
GAMMA_BINS = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.01])

def per_sequence_conformal_inputs(model, ev_split, tr_split):
    ev_split = build_event_features(ev_split)          # freshly simulated frames need features
    pack, _ = pack_sequences(ev_split, tr_split, SCALER, fit=False)
    inf = run_inference(model, pack)
    s = traj_nonconformity(inf["logp_gold"], inf["lens"])
    gh = estimate_gamma(ev_split).set_index("asset_id")["gamma_hat"]
    return pack, inf, s, gh.reindex(pack["ids"]).to_numpy()

# --- calibration pool spanning the degradation range (deployable: no ground truth used) ---
print("Building the degradation-aware calibration pool ...")
cal_s, cal_g = [], []
for gam in np.linspace(0, 1, 6):
    ev_g, tr_g = simulate_dataset(300 if QUICK_RUN else 800, seed=SEED + 1500 + int(gam * 131),
                                  cfg=compound_config(gam))
    _, _, s_g, g_hat = per_sequence_conformal_inputs(MODEL, ev_g, tr_g)
    cal_s.append(s_g); cal_g.append(g_hat)
cal_s = np.concatenate(cal_s); cal_g = np.concatenate(cal_g)

QHAT_MARGINAL = conformal_quantile(cal_s[cal_g <= 0.25], ALPHA)   # calibrated on nominal conditions only
QHAT_BINNED = {}
bin_idx_cal = np.digitize(cal_g, GAMMA_BINS) - 1
for b in range(len(GAMMA_BINS) - 1):
    sel = bin_idx_cal == b
    QHAT_BINNED[b] = conformal_quantile(cal_s[sel], ALPHA) if sel.sum() >= 30 else QHAT_MARGINAL

shift_rows = []
for gam in np.linspace(0, 1, CFG["stress_levels"]):
    ev_g, tr_g = simulate_dataset(CFG["stress_assets"], seed=SEED + 2600 + int(gam * 173),
                                  cfg=compound_config(gam))
    _, inf_g, s_g, g_hat = per_sequence_conformal_inputs(MODEL, ev_g, tr_g)
    bi = np.clip(np.digitize(g_hat, GAMMA_BINS) - 1, 0, len(GAMMA_BINS) - 2)
    q_ad = np.array([QHAT_BINNED[b] for b in bi])
    shift_rows.append(dict(Gamma=gam, GammaHatMean=float(g_hat.mean()),
                           MarginalCoverage=float((s_g <= QHAT_MARGINAL).mean()),
                           AdaptiveCoverage=float((s_g <= q_ad).mean()),
                           MarginalQhat=QHAT_MARGINAL, AdaptiveQhatMean=float(q_ad.mean())))
shift_df = pd.DataFrame(shift_rows)
save_table(shift_df.round(4), "Table_06_conformal_under_shift",
           "Conformal coverage under progressive RFID degradation: marginal vs gamma-adaptive.")
display(shift_df.round(4))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].axhline(1 - ALPHA, ls="--", c="k", lw=1, label=f"nominal {1-ALPHA:.2f}")
ax[0].plot(shift_df["Gamma"], shift_df["MarginalCoverage"], marker="o", label="marginal conformal")
ax[0].plot(shift_df["Gamma"], shift_df["AdaptiveCoverage"], marker="s",
           label=r"$\hat\gamma$-adaptive (Mondrian)")
ax[0].set_xlabel(r"compound degradation severity $\gamma$"); ax[0].set_ylabel("empirical coverage")
ax[0].set_ylim(0, 1.02); ax[0].set_title("Coverage under distribution shift"); ax[0].legend()
ax[1].plot(shift_df["Gamma"], shift_df["GammaHatMean"], marker="o", color="tab:purple")
ax[1].plot([0, 1], [0, 1], "k--", lw=1)
ax[1].set_xlabel(r"true $\gamma$"); ax[1].set_ylabel(r"estimated $\hat\gamma$")
ax[1].set_title("Observable severity estimator")
save_fig(fig, "Fig_04_conformal_shift"); plt.show()

## 9. Certified minimum-cost counterfactual repair (**C3**) and its validation (**C4**)

### 9.1 An exact solution, not a heuristic

Equation (10) asks for the *minimum-cost* edit $X^{cf}$ restoring process consistency. Prior RFID
work (and the first draft of this notebook) answers it greedily — "insert a shortest bridge between
two states". Greedy insertion is neither minimal nor complete: it cannot delete a false read or
correct a misread, and it gives no optimality certificate.

We instead observe that the set of process-consistent routes is exactly the **regular language
$\mathcal{L}(G)$ of source-to-sink paths in the process graph**. Minimum-cost repair is therefore

$$X^{cf}=\arg\min_{P\in\mathcal{L}(G)} d_{\text{edit}}(\hat Z, P),$$

the edit distance from a string to a regular language, which is solved **exactly** by dynamic
programming over (position in the decoded route) × (node of `G`), with intra-column Dijkstra
relaxation for insertions. Cost $O(L\cdot|V|\cdot\deg)$ — milliseconds — and the returned edit set is
*provably* of minimum cost. Each edit carries a physical reading:

| Edit | Physical hypothesis |
|---|---|
| **insert** state `s` | a read at checkpoint `s` was **missed** |
| **delete** observed symbol | the read was a **false / spurious** detection |
| **substitute** `a→b` | the read was **misattributed** to the wrong reader |

### 9.2 Validation: are the counterfactuals *right*?

A counterfactual explanation that is plausible but wrong is worse than none. Because the simulator
retains the injected fault for every event, we can score each proposed edit against the mechanism
that actually fired — **repair identifiability**: precision/recall of recovering the true missed
checkpoints and the true spurious reads. We are not aware of counterfactual RFID explanations having
previously been validated against known corruption provenance.

In [ ]:
# ============================================================
# 9.1 Exact minimum-cost repair: edit distance to the language of admissible paths
# ============================================================
C_INS, C_DEL, C_SUB = 1.0, 1.0, 1.0
BIG = 1e9

def min_cost_repair(obs_route, c_ins=C_INS, c_del=C_DEL, c_sub=C_SUB):
    '''
    Exact minimum-cost edit from `obs_route` to the regular language of admissible
    SOURCE->SINK paths in G. Returns (cost, repaired_path, edits).
    edits: list of ('insert', state, obs_pos) | ('delete', obs_pos, state) | ('sub', obs_pos, a, b)
    '''
    obs = [int(s) for s in obs_route]
    L = len(obs)
    cost = [dict() for _ in range(L + 1)]
    parent = [dict() for _ in range(L + 1)]

    def relax(i, v, c, par):
        if c < cost[i].get(v, BIG) - 1e-12:
            cost[i][v] = c; parent[i][v] = par; return True
        return False

    relax(0, SOURCE, c_ins, ("start_ins", None))
    if L:
        relax(1, SOURCE, 0.0 if obs[0] == SOURCE else c_sub, ("start_match", None))

    for i in range(L + 1):
        # (c) intra-column insertions: Dijkstra over the 8 graph nodes
        frontier = sorted(cost[i].items(), key=lambda kv: kv[1])
        visited = set()
        while frontier:
            v, cv = frontier.pop(0)
            if v in visited or cv > cost[i].get(v, BIG) + 1e-12:
                continue
            visited.add(v)
            for w in ADJ[v]:
                if relax(i, w, cv + c_ins, ("ins", v, i)):
                    frontier.append((w, cost[i][w]))
                    frontier.sort(key=lambda kv: kv[1])
        if i == L:
            break
        for v, cv in list(cost[i].items()):
            # (a) delete the observed symbol (hypothesis: false read)
            relax(i + 1, v, cv + c_del, ("del", v, i))
            # (b) consume the observed symbol by moving to an admissible successor
            for w in ADJ[v]:
                step = 0.0 if obs[i] == w else c_sub
                relax(i + 1, w, cv + step, ("match" if step == 0 else "sub", v, i))

    best_v, best_c = None, BIG
    for v, cv in cost[L].items():
        h = hops_to_sink(v)
        if h == INF:
            continue
        total = cv + c_ins * h
        if total < best_c:
            best_c, best_v = total, v
    if best_v is None:
        return BIG, [], []

    # backtrace
    path_rev, edits = [], []
    v, i = best_v, L
    tail = []
    cur = best_v
    while cur != SINK:                                     # forced insertions to reach the sink
        nxt = min(ADJ[cur], key=lambda w: hops_to_sink(w))
        tail.append(nxt); edits.append(("insert", nxt, L)); cur = nxt
    while True:
        par = parent[i].get(v)
        if par is None:
            break
        kind = par[0]
        if kind == "start_ins":
            path_rev.append(v); edits.append(("insert", v, 0)); break
        if kind == "start_match":
            path_rev.append(v)
            if obs[0] != v: edits.append(("sub", 0, obs[0], v))
            break
        if kind == "ins":
            path_rev.append(v); edits.append(("insert", v, i)); v = par[1]
        elif kind == "del":
            edits.append(("delete", par[2], obs[par[2]])); i -= 1
        elif kind in ("match", "sub"):
            path_rev.append(v)
            if kind == "sub":
                edits.append(("sub", par[2], obs[par[2]], v))
            v = par[1]; i -= 1
        else:
            break
    repaired = list(reversed(path_rev)) + tail
    return float(best_c), repaired, edits

# --- self-test: the routine must be exact and must certify already-valid routes as cost 0 ---
_c, _p, _e = min_cost_repair(list(range(N_STATES)))
assert _c == 0.0 and _p == list(range(N_STATES)) and not _e, (_c, _p, _e)
_c2, _p2, _e2 = min_cost_repair([0, 1, 2, 4, 5, 6, 7])          # one missed checkpoint (S3)
_c3, _p3, _e3 = min_cost_repair([0, 1, 2, 3, 6, 4, 5, 6, 7])    # one spurious read
print("valid route      -> cost", _c)
print("missed S3        -> cost", _c2, "| repaired", _p2, "| edits", _e2)
print("spurious read    -> cost", _c3, "| repaired", _p3, "| edits", _e3)
assert route_is_admissible(_p2) and route_is_admissible(_p3)
print("Exact repair routine verified.")

In [ ]:
# ============================================================
# 9.2 Repair applied to decoded trajectories + identifiability against true corruption
# ============================================================
ev_test_sorted = events[events["split"] == "test"].sort_values(["asset_id", "t"])
EV_BY_ASSET = {aid: g for aid, g in ev_test_sorted.groupby("asset_id", sort=False)}

def repair_report(ids, paths, lens, label):
    rows = []
    for i, aid in enumerate(ids):
        L = int(lens[i])
        dec = collapse(np.asarray(paths[i][:L], int))
        g = EV_BY_ASSET[aid]
        raw = collapse(g["reader"].to_numpy())
        true_route = list(TRAJ_IDX.at[aid, "oracle_true_route"])

        cost_dec, rep_dec, edits = min_cost_repair(dec)
        cost_raw, _, _ = min_cost_repair(raw)

        # ---- identifiability against the corruption mechanism actually injected ----
        observed_states = set(raw)
        truly_missed = set(true_route) - observed_states
        proposed_missed = {s for k, s, _ in [(e[0], e[1], e[2]) for e in edits if e[0] == "insert"]}
        tp_ins = len(proposed_missed & truly_missed)
        n_false_reads = int((g["oracle_src"] == "false").sum())
        n_del = sum(1 for e in edits if e[0] == "delete")

        rows.append(dict(
            asset_id=aid, Method=label, A_s=int(TRAJ_IDX.at[aid, "A_s"]),
            A_o=int(TRAJ_IDX.at[aid, "A_o"]), op_type=TRAJ_IDX.at[aid, "op_type"],
            repair_cost=cost_dec, repair_cost_raw=cost_raw,
            repair_cost_norm=cost_dec / max(len(dec), 1),
            n_edits=len(edits), n_insert=len(proposed_missed), n_delete=n_del,
            n_sub=sum(1 for e in edits if e[0] == "sub"),
            consistent_before=float(graph_inconsistency(dec) == 0 and route_is_admissible(dec)),
            consistent_after=float(route_is_admissible(rep_dec)),
            sim_before=route_similarity(true_route, dec),
            sim_after=route_similarity(true_route, rep_dec),
            obs_distortion=1.0 - route_similarity(raw, dec),      # A_s mechanistic score
            ins_tp=tp_ins, ins_fp=len(proposed_missed) - tp_ins, ins_fn=len(truly_missed) - tp_ins,
            del_prop=n_del, del_true=n_false_reads,
        ))
    return pd.DataFrame(rows)

REPAIR = repair_report(ids_te, INF_TEST["paths"], lens_te, "GT-CRF")
REPAIR_RAW = repair_report(ids_te, paths_from_dict(ids_te, raw_paths, lens_te), lens_te, "RawReader")

def ident_metrics(df, label):
    tp, fp, fn = df["ins_tp"].sum(), df["ins_fp"].sum(), df["ins_fn"].sum()
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return dict(Method=label,
                MissedCheckpointPrecision=prec, MissedCheckpointRecall=rec,
                MissedCheckpointF1=2*prec*rec/max(prec+rec, 1e-9),
                DeletionRateProposed=df["del_prop"].mean(), FalseReadRateTrue=df["del_true"].mean(),
                MeanEdits=df["n_edits"].mean(), MeanRepairCost=df["repair_cost"].mean(),
                ConsistencyBefore=df["consistent_before"].mean(),
                ConsistencyAfter=df["consistent_after"].mean(),
                SimilarityBefore=df["sim_before"].mean(), SimilarityAfter=df["sim_after"].mean())

ident = pd.DataFrame([ident_metrics(REPAIR_RAW, "Repair on raw reader sequence"),
                      ident_metrics(REPAIR, "Repair on GT-CRF decoded trajectory")]).round(4)
save_table(ident, "Table_07_counterfactual_identifiability",
           "Fidelity, sparsity and mechanism-identifiability of minimum-cost counterfactual repair.")
display(ident)

normal_mask = (REPAIR["A_o"] == 0)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].hist([REPAIR.loc[normal_mask, "repair_cost"], REPAIR.loc[~normal_mask, "repair_cost"]],
           bins=np.arange(0, 9) - 0.5, label=["no operational deviation", "operational deviation"],
           density=True)
ax[0].set_xlabel("minimum repair cost (edits)"); ax[0].set_ylabel("density")
ax[0].set_title("Repair cost separates process deviations"); ax[0].legend()
ax[1].scatter(REPAIR["obs_distortion"], REPAIR["repair_cost_norm"], s=6, alpha=0.25,
              c=REPAIR["A_o"], cmap="coolwarm")
ax[1].set_xlabel("observation distortion  (raw vs reconstructed)  -> sensing axis")
ax[1].set_ylabel("normalised irreducible repair cost  -> operational axis")
ax[1].set_title("Mechanistic disentanglement plane")
save_fig(fig, "Fig_05_counterfactual"); plt.show()

In [ ]:
# ============================================================
# 9.3 Worked counterfactual explanations (manuscript-ready examples)
# ============================================================
def verbalise(edits):
    if not edits:
        return "No edit required: the reconstructed route is already process-consistent."
    parts = []
    for e in sorted(edits, key=lambda z: (z[0], str(z[1:]))):
        if e[0] == "insert":
            parts.append(f"a missing read at checkpoint {STATES[e[1]]}")
        elif e[0] == "delete":
            parts.append(f"a spurious read of {STATES[e[2]]} at observation {e[1]}")
        else:
            parts.append(f"a misread of {STATES[e[2]]} that should have been {STATES[e[3]]}")
    return "It suffices to assume " + "; ".join(parts) + " to restore process consistency."

ex_rows = []
for i, aid in enumerate(ids_te):
    L = int(lens_te[i])
    dec = collapse(np.asarray(INF_TEST["paths"][i][:L], int))
    c, rep, edits = min_cost_repair(dec)
    if not edits:
        continue
    g = EV_BY_ASSET[aid]
    ex_rows.append(dict(
        asset_id=aid, true_diagnosis=("operational" if TRAJ_IDX.at[aid, "A_o"] else
                                      ("sensing" if TRAJ_IDX.at[aid, "A_s"] else "normal")),
        op_type=TRAJ_IDX.at[aid, "op_type"],
        observed="->".join(STATES[s] for s in collapse(g["reader"].to_numpy())),
        reconstructed="->".join(STATES[s] for s in dec),
        repaired="->".join(STATES[s] for s in rep),
        repair_cost=c, explanation=verbalise(edits)))
    if len(ex_rows) >= 12:
        break
examples = pd.DataFrame(ex_rows)
save_table(examples, "Table_08_counterfactual_examples",
           "Representative validated counterfactual repairs.")
display(examples.head(8))

## 10. Mechanistic sensing-vs-operational diagnosis (**C5**)

Equation (7) asks for $p(A_s\mid\tilde X,G)$ and $p(A_o\mid\tilde X,G)$. A black-box classifier over
summary statistics can produce those numbers but explains nothing, and — as §3 argued — is the exact
place where corruption ground truth usually leaks in.

We derive the two probabilities from the **geometry of the repair problem** instead, giving two
*label-free* scores that fall out of the reconstruction itself:

* **Sensing axis** — $1-\mathrm{sim}(\text{raw reader route},\ \hat Z)$: how much the observation had
  to be *undone* to obtain a physically coherent story. Distortion in the RFID image.
* **Operational axis** — the normalised **irreducible** minimum repair cost of $\hat Z$: evidence
  that no observation-level edit can rescue, i.e. the physical process really deviated.
* **Evidence-sacrifice gap** — $\frac{1}{T}[\log p(\hat Z_{\text{free}}) - \log p(\hat
  Z_{\text{admissible}})]$: how much likelihood must be given up to force a legal route. A large gap
  means the data insists on an illegal route.

These are compared against the supervised auxiliary neural head and a supervised gradient-boosted
classifier on observable summaries. The interesting result to report is not merely which wins, but
that the *unsupervised mechanistic* scores are competitive — they require no anomaly labels at all,
which matters because operational-anomaly labels are exactly what a real deployment lacks.

In [ ]:
# ============================================================
# 10.1 Evidence-sacrifice gap: free vs process-admissible decoding
# ============================================================
@torch.no_grad()
def emissions_for(model, pack, batch=128):
    out = []
    dl = DataLoader(SeqDS(pack), batch_size=batch, shuffle=False)
    for xn, xr, y, m, ys, yo in dl:
        em, _ = model.encode(xn.to(DEVICE), xr.to(DEVICE), m.to(DEVICE))
        out.append(em.cpu().numpy())
    return np.concatenate(out, 0)

EM_TEST = emissions_for(MODEL, PACK["test"])
A_LEARNED = (MODEL.transitions()).detach().cpu().numpy().astype(np.float64)
A_HARD = np.full((N_STATES, N_STATES), -1e6)
for i in range(N_STATES):
    A_HARD[i, i] = A_LEARNED[i, i]
for a, b in VALID_EDGES:
    A_HARD[a, b] = A_LEARNED[a, b]

def viterbi_score(logem, A):
    n, S = logem.shape
    dp = logem[0].copy()
    for t in range(1, n):
        dp = (dp[:, None] + A).max(0) + logem[t]
    return float(dp.max())

gap = np.zeros(len(ids_te))
for i in range(len(ids_te)):
    L = int(lens_te[i]); em = EM_TEST[i, :L].astype(np.float64)
    gap[i] = (viterbi_score(em, A_LEARNED) - viterbi_score(em, A_HARD)) / max(L, 1)

REPAIR["evidence_gap"] = gap
REPAIR["neural_p_As"] = INF_TEST["diag"][:, 0]
REPAIR["neural_p_Ao"] = INF_TEST["diag"][:, 1]
print("Evidence-sacrifice gap computed.")

In [ ]:
# ============================================================
# 10.2 Dual anomaly diagnosis: mechanistic (label-free) vs supervised
# ============================================================
from sklearn.ensemble import HistGradientBoostingClassifier

sev_test = estimate_gamma(ev_test_sorted).set_index("asset_id").reindex(ids_te)
SUP_FEATURES = ["obs_distortion", "repair_cost_norm", "evidence_gap", "n_edits",
                "n_insert", "n_delete", "n_sub"]
SUP_X = np.hstack([REPAIR[SUP_FEATURES].to_numpy(),
                   sev_test[["gamma_hat"]].to_numpy()])
assert_no_leakage(SUP_FEATURES + ["gamma_hat"], "diagnosis feature set")

# supervised diagnostic models are fitted on the CALIBRATION split (never on test)
def diagnosis_frame(model, split):
    ev_s = events[events["split"] == split].sort_values(["asset_id", "t"])
    pack = PACK[split]
    inf = INF_CAL if split == "cal" else run_inference(model, pack)
    ids = pack["ids"]; lens = pack["lens"]
    by_asset = {a: g for a, g in ev_s.groupby("asset_id", sort=False)}
    rows = []
    em_all = emissions_for(model, pack)
    for i, aid in enumerate(ids):
        L = int(lens[i]); dec = collapse(np.asarray(inf["paths"][i][:L], int))
        raw = collapse(by_asset[aid]["reader"].to_numpy())
        c, rep, edits = min_cost_repair(dec)
        em = em_all[i, :L].astype(np.float64)
        g_ = (viterbi_score(em, A_LEARNED) - viterbi_score(em, A_HARD)) / max(L, 1)
        rows.append(dict(asset_id=aid, obs_distortion=1 - route_similarity(raw, dec),
                         repair_cost_norm=c / max(len(dec), 1), evidence_gap=g_,
                         n_edits=len(edits),
                         n_insert=sum(1 for e in edits if e[0] == "insert"),
                         n_delete=sum(1 for e in edits if e[0] == "delete"),
                         n_sub=sum(1 for e in edits if e[0] == "sub"),
                         A_s=int(TRAJ_IDX.at[aid, "A_s"]), A_o=int(TRAJ_IDX.at[aid, "A_o"])))
    df = pd.DataFrame(rows)
    gh = estimate_gamma(ev_s).set_index("asset_id").reindex(df["asset_id"])
    df["gamma_hat"] = gh["gamma_hat"].to_numpy()
    return df

DIAG_CAL = diagnosis_frame(MODEL, "cal")
X_cal_d = DIAG_CAL[SUP_FEATURES + ["gamma_hat"]].to_numpy()
CLF_S = HistGradientBoostingClassifier(random_state=SEED).fit(X_cal_d, DIAG_CAL["A_s"])
CLF_O = HistGradientBoostingClassifier(random_state=SEED).fit(X_cal_d, DIAG_CAL["A_o"])

y_s, y_o = REPAIR["A_s"].to_numpy(), REPAIR["A_o"].to_numpy()
scores = {
    "Mechanistic: observation distortion":      (REPAIR["obs_distortion"].to_numpy(), "A_s"),
    "Mechanistic: irreducible repair cost":     (REPAIR["repair_cost_norm"].to_numpy(), "A_o"),
    "Mechanistic: evidence-sacrifice gap":      (REPAIR["evidence_gap"].to_numpy(), "A_o"),
    "Supervised: neural auxiliary head":        (REPAIR["neural_p_As"].to_numpy(), "A_s"),
    "Supervised: neural auxiliary head (Ao)":   (REPAIR["neural_p_Ao"].to_numpy(), "A_o"),
    "Supervised: GBM on repair features":       (CLF_S.predict_proba(SUP_X)[:, 1], "A_s"),
    "Supervised: GBM on repair features (Ao)":  (CLF_O.predict_proba(SUP_X)[:, 1], "A_o"),
}
rows = []
for name, (sc, target) in scores.items():
    yy = y_s if target == "A_s" else y_o
    rows.append(dict(Score=name, Target=target, LabelsRequired=("no" if name.startswith("Mech") else "yes"),
                     AUROC=roc_auc_score(yy, sc), AUPRC=average_precision_score(yy, sc)))
diag_tab = pd.DataFrame(rows).round(4)
save_table(diag_tab, "Table_09_dual_diagnosis",
           "Sensing vs operational anomaly diagnosis: mechanistic and supervised scores.")
display(diag_tab)

# joint 2x2 confusion of the deployed decision rule
thr_s = np.quantile(DIAG_CAL["obs_distortion"], 1 - DIAG_CAL["A_s"].mean())
thr_o = np.quantile(DIAG_CAL["repair_cost_norm"], 1 - DIAG_CAL["A_o"].mean())
pred_s = (REPAIR["obs_distortion"].to_numpy() >= thr_s).astype(int)
pred_o = (REPAIR["repair_cost_norm"].to_numpy() >= thr_o).astype(int)
joint = pd.crosstab(pd.Series([f"As={a},Ao={b}" for a, b in zip(y_s, y_o)], name="True"),
                    pd.Series([f"As={a},Ao={b}" for a, b in zip(pred_s, pred_o)], name="Predicted"))
save_table(joint.reset_index(), "Table_10_joint_diagnosis_confusion",
           "Joint sensing/operational diagnosis confusion (mechanistic rule).")
display(joint)

fig, ax = plt.subplots(figsize=(5.4, 4.6))
im = ax.imshow(joint.to_numpy(), cmap="Blues")
ax.set_xticks(range(joint.shape[1]), joint.columns, rotation=30, ha="right")
ax.set_yticks(range(joint.shape[0]), joint.index)
for i in range(joint.shape[0]):
    for j in range(joint.shape[1]):
        ax.text(j, i, joint.to_numpy()[i, j], ha="center", va="center", fontsize=8)
ax.set_title("Joint dual-anomaly diagnosis"); fig.colorbar(im, ax=ax, fraction=0.046)
save_fig(fig, "Fig_06_diagnosis_confusion"); plt.show()

## 11. Risk-controlled selective decisions (**C6**, second half)

A risk–coverage curve describes behaviour; it does not *guarantee* anything. Because we hold out a
calibration split, we can do better and select the abstention threshold by **Learn-then-Test**: over
a grid of thresholds $\lambda$, test $H_0:\ R(\lambda)>\epsilon$ with a Hoeffding bound, apply a
Bonferroni correction across the grid, and deploy the *most permissive* threshold whose null is
rejected. The result is a finite-sample statement of the form

> with probability $\ge 1-\delta$, the selective reconstruction error among autonomously accepted
> trajectories is at most $\epsilon$.

The policy then maps uncertainty and diagnosis onto the manuscript's action set
$\mathcal{A}=\{\text{accept, reconstruct, re-read, flag, human verification}\}$ (Eq. 9), and the
operational utility of Eq. (11) is swept over cost weights rather than asserted at one point.

In [ ]:
# ============================================================
# 11.1 Learn-then-Test threshold selection with a finite-sample risk guarantee
# ============================================================
def seq_uncertainty(inf, i):
    '''Sequence uncertainty: mean posterior entropy of the CRF marginals over the observed events.'''
    L = int(inf["lens"][i]); p = inf["marg"][i, :L]
    return float((-(p * np.log(np.clip(p, 1e-12, 1))).sum(1)).mean())

def exact_route_correct(inf, ids, lens):
    out = np.zeros(len(ids))
    for i, aid in enumerate(ids):
        dec = collapse(np.asarray(inf["paths"][i][:int(lens[i])], int))
        out[i] = float(list(dec) == list(TRAJ_IDX.at[aid, "oracle_true_route"]))
    return out

U_CAL = np.array([seq_uncertainty(INF_CAL, i) for i in range(len(INF_CAL["ids"]))])
U_TEST = np.array([seq_uncertainty(INF_TEST, i) for i in range(len(INF_TEST["ids"]))])
OK_CAL = exact_route_correct(INF_CAL, PACK["cal"]["ids"], PACK["cal"]["lens"])
OK_TEST = exact_route_correct(INF_TEST, ids_te, lens_te)

def hoeffding_pvalue(risk_hat, n, eps):
    '''p-value for H0: R > eps, using the one-sided Hoeffding bound on a [0,1] loss.'''
    if risk_hat >= eps or n == 0:
        return 1.0
    return float(np.exp(-2 * n * (eps - risk_hat) ** 2))

EPS, DELTA = CFG["risk_epsilon"], CFG["risk_delta"]
grid = np.quantile(U_CAL, np.linspace(0.05, 1.0, 40))
ltt_rows = []
for lam in grid:
    keep = U_CAL <= lam
    n = int(keep.sum())
    if n < 30:
        continue
    r = 1.0 - OK_CAL[keep].mean()
    p = hoeffding_pvalue(r, n, EPS)
    ltt_rows.append(dict(Threshold=lam, Coverage=keep.mean(), CalRisk=r, n=n, pvalue=p,
                         Rejected=bool(p <= DELTA / len(grid))))
ltt = pd.DataFrame(ltt_rows)
valid = ltt[ltt["Rejected"]]
LAMBDA_STAR = float(valid["Threshold"].max()) if len(valid) else float(grid[0])
save_table(ltt.round(5), "Table_11_learn_then_test",
           "Learn-then-Test selection of the abstention threshold with Bonferroni correction.")

keep_te = U_TEST <= LAMBDA_STAR
guarantee = pd.DataFrame([dict(
    Epsilon=EPS, Delta=DELTA, GridSize=len(grid), LambdaStar=LAMBDA_STAR,
    TestCoverage=float(keep_te.mean()),
    TestSelectiveRisk=float(1 - OK_TEST[keep_te].mean()) if keep_te.sum() else np.nan,
    UnconditionalRisk=float(1 - OK_TEST.mean()),
    GuaranteeHolds=bool(keep_te.sum() and (1 - OK_TEST[keep_te].mean()) <= EPS))]).round(4)
save_table(guarantee, "Table_12_risk_guarantee",
           "Finite-sample selective-risk guarantee validated on the test partition.")
display(guarantee)

# risk-coverage curve for the manuscript
rc_rows = []
for q in np.linspace(0.05, 1.0, 40):
    lam = np.quantile(U_TEST, q); k = U_TEST <= lam
    if k.sum() < 20: continue
    rc_rows.append(dict(Coverage=k.mean(), SelectiveRisk=1 - OK_TEST[k].mean()))
rc = pd.DataFrame(rc_rows)
_trapz = getattr(np, "trapezoid", None) or np.trapz   # numpy 1.x / 2.x compatibility
aurc = float(_trapz(rc["SelectiveRisk"], rc["Coverage"]) / (rc["Coverage"].max() - rc["Coverage"].min()))
save_table(rc.round(4), "Table_13_risk_coverage", f"Risk-coverage curve (AURC={aurc:.4f}).")

fig, ax = plt.subplots(figsize=(5.8, 4.4))
ax.plot(rc["Coverage"], rc["SelectiveRisk"], marker="o", ms=3)
ax.axhline(EPS, ls="--", c="tab:red", label=f"risk budget $\\epsilon$={EPS}")
ax.axvline(float(keep_te.mean()), ls=":", c="tab:green", label="LTT-selected coverage")
ax.set_xlabel("coverage (fraction decided autonomously)"); ax.set_ylabel("selective reconstruction risk")
ax.set_title(f"Risk-coverage (AURC={aurc:.3f})"); ax.legend()
save_fig(fig, "Fig_07_risk_coverage"); plt.show()

In [ ]:
# ============================================================
# 11.2 Five-action decision policy and the operational utility of Eq. (11)
# ============================================================
ACTIONS = ["accept", "reconstruct", "re-read", "flag", "human_verification"]

def policy(u, p_As, p_Ao, lam_star, u_hi, thr_s=0.5, thr_o=0.5):
    if p_Ao >= thr_o:                       return "flag"
    if u > u_hi:                            return "human_verification"
    if u > lam_star:                        return "re-read"
    if p_As >= thr_s:                       return "reconstruct"
    return "accept"

U_HI = float(np.quantile(U_CAL, 0.95))
p_As_te = CLF_S.predict_proba(SUP_X)[:, 1]
p_Ao_te = CLF_O.predict_proba(SUP_X)[:, 1]
acts = np.array([policy(U_TEST[i], p_As_te[i], p_Ao_te[i], LAMBDA_STAR, U_HI)
                 for i in range(len(ids_te))])

action_mix = pd.DataFrame({"Action": ACTIONS,
                           "Share": [float((acts == a).mean()) for a in ACTIONS]})
save_table(action_mix.round(4), "Table_14_action_distribution", "Distribution of selected actions.")
display(action_mix.round(4))

def utility(actions, ok, y_o, l_false, l_miss, l_verify):
    auto = np.isin(actions, ["accept", "reconstruct"])
    benefit = (auto & (ok == 1)).mean()
    false_alarm = (np.isin(actions, ["flag"]) & (y_o == 0)).mean()
    missed = (auto & (y_o == 1)).mean()
    verify = np.isin(actions, ["re-read", "human_verification"]).mean()
    return benefit - l_false*false_alarm - l_miss*missed - l_verify*verify

policies = {
    "Proposed (uncertainty + dual diagnosis)": acts,
    "Always accept": np.array(["accept"] * len(ids_te)),
    "Always verify": np.array(["human_verification"] * len(ids_te)),
    "Confidence-only abstention": np.where(U_TEST <= LAMBDA_STAR, "accept", "human_verification"),
}
util_rows = []
for lf in [0.5, 1.0, 2.0]:
    for lm in [1.0, 2.0, 4.0]:
        for lv in [0.15, 0.35]:
            for name, a in policies.items():
                util_rows.append(dict(Policy=name, LambdaFalseAlarm=lf, LambdaMissed=lm,
                                      LambdaVerify=lv,
                                      Utility=utility(a, OK_TEST, y_o, lf, lm, lv)))
util = pd.DataFrame(util_rows)
util_summary = (util.groupby("Policy")["Utility"].agg(["mean", "min", "max"])
                .reset_index().rename(columns={"mean": "MeanUtility", "min": "WorstCase",
                                               "max": "BestCase"}).round(4)
                .sort_values("MeanUtility", ascending=False))
save_table(util, "Table_15_utility_sensitivity", "Operational utility over the cost-weight grid.")
save_table(util_summary, "Table_16_utility_summary", "Utility summary by decision policy.")
display(util_summary)

fig, ax = plt.subplots(figsize=(7.4, 4.2))
for name in policies:
    d = util[util["Policy"] == name].sort_values("LambdaMissed")
    m = d.groupby("LambdaMissed")["Utility"].mean()
    ax.plot(m.index, m.values, marker="o", label=name)
ax.set_xlabel(r"cost of a missed operational anomaly $\lambda_2$"); ax.set_ylabel("operational utility $U$")
ax.set_title("Utility sensitivity to decision costs"); ax.legend(fontsize=8)
save_fig(fig, "Fig_08_utility"); plt.show()

## 12. Stress testing under progressive RFID degradation

The model is trained **once**, on nominal conditions, and evaluated out-of-distribution across each
corruption axis independently and then compounded. The question is not peak accuracy at a nominal
operating point — it is whether performance degrades *gracefully*, and whether the process graph buys
robustness precisely where the evidence gets thin.

In [ ]:
# ============================================================
# 12. Single-factor and compound stress tests
# ============================================================
STRESS_AXES = {
    "MissingReads":  ("p_miss",        np.linspace(0.00, 0.55, CFG["stress_levels"])),
    "Duplicates":    ("p_dup",         np.linspace(0.00, 0.55, CFG["stress_levels"])),
    "FalseReads":    ("p_false",       np.linspace(0.00, 0.35, CFG["stress_levels"])),
    "RSSINoise":     ("rssi_noise",    np.linspace(0.5, 12.0, CFG["stress_levels"])),
    "TimingDelay":   ("p_delay",       np.linspace(0.00, 0.50, CFG["stress_levels"])),
    "ReaderOutage":  ("p_reader_fail", np.linspace(0.00, 0.35, CFG["stress_levels"])),
}

def evaluate_under(cfg, seed, n_assets=None):
    n_assets = n_assets or CFG["stress_assets"]
    ev_s, tr_s = simulate_dataset(n_assets, seed=seed, cfg=cfg)
    ev_s = build_event_features(ev_s)
    pack, _ = pack_sequences(ev_s, tr_s, SCALER, fit=False)
    tr_idx = tr_s.set_index("asset_id")
    inf = run_inference(MODEL, pack)
    by_asset = {a: g for a, g in ev_s.sort_values(["asset_id", "t"]).groupby("asset_id", sort=False)}

    sims_p, sims_r, exact_p, ginc_p, repair_c = [], [], [], [], []
    for i, aid in enumerate(pack["ids"]):
        L = int(pack["lens"][i])
        dec = collapse(np.asarray(inf["paths"][i][:L], int))
        raw = collapse(by_asset[aid]["reader"].to_numpy())
        true_route = list(tr_idx.at[aid, "oracle_true_route"])
        sims_p.append(route_similarity(true_route, dec))
        sims_r.append(route_similarity(true_route, raw))
        exact_p.append(float(list(dec) == true_route))
        ginc_p.append(graph_inconsistency(dec))
        if i < 250:
            repair_c.append(min_cost_repair(dec)[0])
    return dict(ProposedSimilarity=np.mean(sims_p), RawReaderSimilarity=np.mean(sims_r),
                ExactRouteRate=np.mean(exact_p), GraphInconsistency=np.mean(ginc_p),
                MeanRepairCost=np.mean(repair_c) if repair_c else np.nan,
                SensingAUROC=roc_auc_score(tr_s["A_s"], inf["diag"][:, 0]) if tr_s["A_s"].nunique() > 1 else np.nan,
                OperationalAUROC=roc_auc_score(tr_s["A_o"], inf["diag"][:, 1]) if tr_s["A_o"].nunique() > 1 else np.nan)

stress_rows = []
t0 = time.time()
for axis, (param, levels) in STRESS_AXES.items():
    for lv in levels:
        cfg = CorruptionConfig(**{**asdict(NOMINAL), param: float(lv)})
        r = evaluate_under(cfg, seed=SEED + 3000 + int(lv * 1000) + (sum(ord(ch) for ch in axis) % 500))
        r.update(Axis=axis, Level=float(lv)); stress_rows.append(r)
stress = pd.DataFrame(stress_rows)
save_table(stress.round(4), "Table_17_stress_single_factor",
           "Single-factor RFID degradation stress tests.")
print(f"Single-factor stress sweep in {time.time()-t0:.0f}s")

fig, axes = plt.subplots(2, 3, figsize=(14, 7.6))
for ax, axis in zip(axes.ravel(), STRESS_AXES):
    d = stress[stress["Axis"] == axis].sort_values("Level")
    ax.plot(d["Level"], d["RawReaderSimilarity"], marker="o", label="raw reader sequence")
    ax.plot(d["Level"], d["ProposedSimilarity"], marker="s", label="GT-CRF (proposed)")
    ax.plot(d["Level"], d["ExactRouteRate"], marker="^", ls="--", label="exact route rate")
    ax.set_title(axis); ax.set_xlabel("degradation level"); ax.set_ylim(0, 1.02)
    ax.set_ylabel("trajectory similarity")
axes.ravel()[0].legend(fontsize=8)
save_fig(fig, "Fig_09_stress_single_factor"); plt.show()

compound_rows = []
for gam in np.linspace(0, 1, CFG["stress_levels"]):
    r = evaluate_under(compound_config(gam), seed=SEED + 4000 + int(gam * 211))
    r["Gamma"] = float(gam); compound_rows.append(r)
compound = pd.DataFrame(compound_rows)
save_table(compound.round(4), "Table_18_stress_compound", "Compound degradation stress test.")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))
ax[0].plot(compound["Gamma"], compound["RawReaderSimilarity"], marker="o", label="raw reader")
ax[0].plot(compound["Gamma"], compound["ProposedSimilarity"], marker="s", label="GT-CRF")
ax[0].plot(compound["Gamma"], compound["ExactRouteRate"], marker="^", ls="--", label="exact route")
ax[0].set_xlabel(r"compound severity $\gamma$"); ax[0].set_ylabel("similarity / rate")
ax[0].set_title("Graceful degradation"); ax[0].legend(fontsize=8); ax[0].set_ylim(0, 1.02)
ax[1].plot(compound["Gamma"], compound["SensingAUROC"], marker="o", label=r"$p(A_s)$ AUROC")
ax[1].plot(compound["Gamma"], compound["OperationalAUROC"], marker="s", label=r"$p(A_o)$ AUROC")
ax[1].axhline(0.5, ls="--", c="k", lw=1)
ax[1].set_xlabel(r"compound severity $\gamma$"); ax[1].set_ylabel("AUROC")
ax[1].set_title("Diagnosis under degradation"); ax[1].legend(fontsize=8)
save_fig(fig, "Fig_10_stress_compound"); plt.show()

## 13. Ablations and repeated-seed statistical validation

Each component of the framework is removed in turn. Every comparison is then repeated over
independent simulation seeds and tested with a **paired Wilcoxon signed-rank test corrected by
Holm–Bonferroni**, reported together with a **Cliff's δ effect size** and a bootstrap confidence
interval. A p-value without an effect size is not evidence of a meaningful improvement, and an
uncorrected family of comparisons is not evidence at all.

In [ ]:
# ============================================================
# 13.1 Component ablations
# ============================================================
ABLATIONS = []

def ablate_row(name, ids, paths, lens, note=""):
    r = evaluate_paths(ids, paths, lens, name); r["Note"] = note
    return r

ABLATIONS.append(ablate_row("Full framework (GT-CRF)", ids_te, INF_TEST["paths"], lens_te,
                            "graph in objective + CRF + calibration + repair"))
ABLATIONS.append(ablate_row("- process graph G", ids_te, INF_TNG["paths"], lens_te,
                            "free transition potentials"))
ABLATIONS.append(ablate_row("- attention (GRU encoder, no G)", ids_te, INF_GRU["paths"], lens_te,
                            "recurrent temporal encoder"))
ABLATIONS.append(ablate_row("- structured decoding (per-event RF)", ids_te,
                            paths_from_dict(ids_te, rf_paths, lens_te), lens_te,
                            "no temporal coupling"))
ABLATIONS.append(ablate_row("- end-to-end structure (RF + post-hoc Viterbi)", ids_te,
                            paths_from_dict(ids_te, RFV_PATHS, lens_te), lens_te,
                            "graph used at inference only"))
ABLATIONS.append(ablate_row("- learning (raw reader stream)", ids_te,
                            paths_from_dict(ids_te, raw_paths, lens_te), lens_te, "no model"))

abl = pd.DataFrame(ABLATIONS)

# effect of counterfactual repair on top of the full framework
abl_repair = pd.DataFrame([dict(
    Method="Full framework + counterfactual repair", StateAccuracy=np.nan, StateMacroF1=np.nan,
    TrajSimilarity=REPAIR["sim_after"].mean(), ExactRouteRate=np.nan,
    GraphInconsistency=1.0 - REPAIR["consistent_after"].mean(),
    Note="minimum-cost repair applied to the decoded route")])
abl = pd.concat([abl, abl_repair], ignore_index=True)
save_table(abl.round(4), "Table_19_ablation", "Component ablation of the proposed framework.")
display(abl.round(4))

fig, ax = plt.subplots(figsize=(8.4, 4.4))
d = abl.dropna(subset=["TrajSimilarity"]).sort_values("TrajSimilarity")
ax.barh(d["Method"], d["TrajSimilarity"],
        color=["tab:red" if "Full" in m else "tab:blue" for m in d["Method"]])
ax.set_xlabel("mean trajectory similarity"); ax.set_xlim(0, 1)
ax.set_title("Ablation: contribution of each component")
save_fig(fig, "Fig_11_ablation"); plt.show()

In [ ]:
# ============================================================
# 13.2 Repeated-seed experiments with corrected paired statistics
# ============================================================
from scipy.stats import wilcoxon

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    n = min(len(x), 4000)
    xs, ys = x[:n], y[:n]
    gt = sum((xs[:, None] > ys[None, :]).sum(1))
    lt = sum((xs[:, None] < ys[None, :]).sum(1))
    return float((gt - lt) / (len(xs) * len(ys)))

def bootstrap_ci(v, n_boot=2000, alpha=0.05, seed=7):
    rng = np.random.default_rng(seed); v = np.asarray(v, float)
    m = [rng.choice(v, len(v), replace=True).mean() for _ in range(n_boot)]
    return float(np.quantile(m, alpha/2)), float(np.quantile(m, 1-alpha/2))

def holm(pvals):
    order = np.argsort(pvals); n = len(pvals); adj = np.empty(n)
    running = 0.0
    for rank, idx in enumerate(order):
        val = (n - rank) * pvals[idx]
        running = max(running, val)
        adj[idx] = min(1.0, running)
    return adj

print("Running repeated-seed validation ...")
seed_rows, paired = [], defaultdict(list)
for rep in range(CFG["n_seeds"]):
    sd = SEED + 7000 + rep * 17
    ev_r, tr_r = simulate_dataset(1200 if QUICK_RUN else 3000, seed=sd, cfg=NOMINAL)
    ev_r = build_event_features(ev_r)
    pack_r, _ = pack_sequences(ev_r, tr_r, SCALER, fit=False)
    tri = tr_r.set_index("asset_id")
    inf_r = run_inference(MODEL, pack_r)
    by_a = {a: g for a, g in ev_r.sort_values(["asset_id", "t"]).groupby("asset_id", sort=False)}

    Xr, _, ev_flat = (None, None, None)
    Xr_num = ev_r.sort_values(["asset_id", "t"])
    Xr_feats = Xr_num[FEATS].to_numpy(np.float64)
    Xr_oh = pd.get_dummies(pd.Categorical(Xr_num["reader"], categories=range(N_STATES))).to_numpy(float)
    rf_proba_r = RF.predict_proba(np.hstack([Xr_feats, Xr_oh]))
    logem_r = np.log(np.clip(rf_proba_r, 1e-12, 1))

    per_prop, per_rfv, per_raw = [], [], []
    idx = 0
    for i, aid in enumerate(pack_r["ids"]):
        g = by_a[aid]; L = len(g)
        true_route = list(tri.at[aid, "oracle_true_route"])
        dec = collapse(np.asarray(inf_r["paths"][i][:int(pack_r["lens"][i])], int))
        rfv = collapse(viterbi_numpy(logem_r[idx:idx+L], TRANS_LOG_NP))
        raw = collapse(g["reader"].to_numpy())
        idx += L
        per_prop.append(route_similarity(true_route, dec))
        per_rfv.append(route_similarity(true_route, rfv))
        per_raw.append(route_similarity(true_route, raw))

    seed_rows.append(dict(Seed=sd, Proposed=np.mean(per_prop), RFViterbi=np.mean(per_rfv),
                          RawReader=np.mean(per_raw),
                          SensingAUROC=roc_auc_score(tr_r["A_s"], inf_r["diag"][:, 0]),
                          OperationalAUROC=roc_auc_score(tr_r["A_o"], inf_r["diag"][:, 1])))
    paired["Proposed vs RF+Viterbi"].append((np.array(per_prop), np.array(per_rfv)))
    paired["Proposed vs RawReader"].append((np.array(per_prop), np.array(per_raw)))

seeds_df = pd.DataFrame(seed_rows)
seed_summary = seeds_df.drop(columns="Seed").agg(["mean", "std"]).T.reset_index()
seed_summary.columns = ["Metric", "Mean", "SD"]
seed_summary[["CI_low", "CI_high"]] = [bootstrap_ci(seeds_df[m]) for m in seed_summary["Metric"]]
save_table(seeds_df.round(4), "Table_20_repeated_seeds", "Per-seed results.")
save_table(seed_summary.round(4), "Table_21_repeated_seed_summary",
           "Repeated-seed means with bootstrap 95% confidence intervals.")
display(seed_summary.round(4))

stat_rows, pvals = [], []
for name, pairs in paired.items():
    a = np.concatenate([p[0] for p in pairs]); b = np.concatenate([p[1] for p in pairs])
    try:
        stat, p = wilcoxon(a, b, zero_method="zsplit")
    except ValueError:
        stat, p = np.nan, 1.0
    lo, hi = bootstrap_ci(a - b)
    stat_rows.append(dict(Comparison=name, MeanDifference=float((a-b).mean()),
                          CI_low=lo, CI_high=hi, CliffsDelta=cliffs_delta(a, b),
                          WilcoxonP=float(p), N=len(a)))
    pvals.append(float(p))
stats_df = pd.DataFrame(stat_rows)
stats_df["HolmAdjustedP"] = holm(np.array(pvals))
stats_df["Significant(0.05)"] = stats_df["HolmAdjustedP"] < 0.05
save_table(stats_df.round(6), "Table_22_paired_statistics",
           "Paired comparisons with Holm-Bonferroni correction and Cliff's delta effect sizes.")
display(stats_df.round(5))

In [ ]:
# ============================================================
# 13.3 Computational cost (operational feasibility)
# ============================================================
def time_call(fn, n=3):
    ts = []
    for _ in range(n):
        t0 = time.time(); fn(); ts.append(time.time() - t0)
    return float(np.median(ts))

small = {k: (v[:256] if isinstance(v, np.ndarray) else v) for k, v in PACK["test"].items()}
small["ids"] = PACK["test"]["ids"][:256]
t_inf = time_call(lambda: run_inference(MODEL, small), 3)
sample_routes = [collapse(np.asarray(INF_TEST["paths"][i][:int(lens_te[i])], int))
                 for i in range(min(256, len(ids_te)))]
t_rep = time_call(lambda: [min_cost_repair(r) for r in sample_routes], 3)

cost_tab = pd.DataFrame([
    dict(Component="GT-CRF training (once)", Seconds=TRAIN_SECS, PerTrajectory_ms=np.nan),
    dict(Component="RF baseline training (once)", Seconds=RF_SECS, PerTrajectory_ms=np.nan),
    dict(Component="GT-CRF inference (decode + marginals)", Seconds=t_inf,
         PerTrajectory_ms=1000 * t_inf / 256),
    dict(Component="Exact counterfactual repair", Seconds=t_rep,
         PerTrajectory_ms=1000 * t_rep / len(sample_routes)),
]).round(4)
save_table(cost_tab, "Table_23_computational_cost", "Computational cost on the current runtime.")
display(cost_tab)
print("Device:", DEVICE)

## 14. Positioning: what is actually new here

This table is written for the *Comparative Methodological Positioning* subsection of the manuscript.
Each row states a capability, what conventional RFID analytics does, and what this framework does —
so the novelty claim rests on demonstrated behaviour in the tables above rather than on assertion.

In [ ]:
# ============================================================
# 14. Novelty positioning table (auto-populated with measured evidence)
# ============================================================
def g(df, q, col, default=np.nan):
    try:
        v = df.query(q)[col]
        return float(v.iloc[0]) if len(v) else default
    except Exception:
        return default

prop_sim = g(recon, "Method=='GT-CRF (proposed)'", "TrajSimilarity")
rfv_sim  = g(recon, "Method=='RF+Viterbi'", "TrajSimilarity")
raw_sim  = g(recon, "Method=='RawReader'", "TrajSimilarity")
ident_f1 = g(ident, "Method=='Repair on GT-CRF decoded trajectory'", "MissedCheckpointF1")
prop_gi  = g(recon, "Method=='GT-CRF (proposed)'", "GraphInconsistency")

positioning = pd.DataFrame([
    dict(Capability="Target of inference",
         Conventional="classify each read as normal/anomalous",
         ThisWork="reconstruct the latent trajectory p(z_t | x_1:t, G)",
         Evidence=f"trajectory similarity {prop_sim:.3f} vs {raw_sim:.3f} for the raw stream"),
    dict(Capability="Use of process knowledge",
         Conventional="post-hoc filtering / rule check",
         ThisWork="graph potentials inside the training objective (Eq. 2)",
         Evidence=f"vs RF+Viterbi (graph at inference only): {prop_sim:.3f} vs {rfv_sim:.3f}"),
    dict(Capability="Anomaly semantics",
         Conventional="single undifferentiated anomaly score",
         ThisWork="separate p(A_s) and p(A_o) with a mechanistic derivation",
         Evidence=f"see Table_09; label-free scores included"),
    dict(Capability="Counterfactual explanation",
         Conventional="absent, or heuristic gap-filling",
         ThisWork="exact minimum-cost edit to the language of admissible paths",
         Evidence=f"consistency restored on {REPAIR['consistent_after'].mean()*100:.1f}% of trajectories"),
    dict(Capability="Explanation validation",
         Conventional="not evaluated",
         ThisWork="edits scored against the injected corruption mechanism",
         Evidence=f"missed-checkpoint F1 {ident_f1:.3f}"),
    dict(Capability="Uncertainty guarantee",
         Conventional="softmax confidence, or marginal conformal assumed valid",
         ThisWork="degradation-adaptive conformal + Learn-then-Test risk control",
         Evidence="see Table_06 (coverage under shift) and Table_12 (finite-sample guarantee)"),
    dict(Capability="Decision layer",
         Conventional="fixed threshold",
         ThisWork="five-action policy with cost-sensitivity analysis",
         Evidence="see Table_16"),
    dict(Capability="Evaluation integrity",
         Conventional="corruption flags frequently used as input features",
         ThisWork="machine-checked leakage firewall",
         Evidence="Table_00; assert_no_leakage() guards every design matrix"),
])
save_table(positioning, "Table_24_novelty_positioning",
           "Comparative methodological positioning with measured evidence.")
display(positioning)

## 15. Claim → evidence map

Every claim the manuscript can make is bound here to the artefact that supports it. Numbers should be
copied into the paper **only** from these files, never re-typed from a notebook cell output.

In [ ]:
# ============================================================
# 15. Manuscript claim -> evidence map
# ============================================================
claims = pd.DataFrame([
    ("4.1 Experimental configuration", "Corpus, partitions, prevalence",
     "Table_01_partitions.csv, config.json"),
    ("4.1 Baseline performance", "Reconstruction vs 6 comparators",
     "Table_03_reconstruction.csv, Fig_02_reconstruction"),
    ("4.2 Robustness", "Single-factor + compound degradation",
     "Table_17_stress_single_factor.csv, Table_18_stress_compound.csv, Fig_09, Fig_10"),
    ("4.3 Sensing vs operational diagnosis", "Dual anomaly scores, joint confusion",
     "Table_09_dual_diagnosis.csv, Table_10_joint_diagnosis_confusion.csv, Fig_06"),
    ("4.4 Uncertainty", "Calibration, conformal, coverage under shift",
     "Table_04_calibration.csv, Table_05_conformal.csv, Table_06_conformal_under_shift.csv, Fig_03, Fig_04"),
    ("4.4 Selective prediction", "LTT guarantee, risk-coverage, action mix",
     "Table_11, Table_12, Table_13, Table_14, Fig_07"),
    ("4.4 Counterfactual repair", "Fidelity, sparsity, identifiability, examples",
     "Table_07_counterfactual_identifiability.csv, Table_08_counterfactual_examples.csv, Fig_05"),
    ("4.5 Ablation", "Component contributions", "Table_19_ablation.csv, Fig_11"),
    ("4.5 Statistical analysis", "Repeated seeds, Holm-corrected Wilcoxon, Cliff's delta",
     "Table_20, Table_21, Table_22"),
    ("4.5 Operational utility", "Cost-weight sensitivity", "Table_15, Table_16, Fig_08"),
    ("4.6.1 Positioning", "What is new and the evidence for it", "Table_24_novelty_positioning.csv"),
    ("4.6.2 Limitations", "Scope and threats to validity", "Limitations.md"),
    ("3.11 Reproducibility", "Config hash, environment, cost",
     "config.json, Table_23_computational_cost.csv, Results_Inventory.csv"),
], columns=["Manuscript section", "Claim", "Supporting artefact"])
save_table(claims, "Table_25_claim_evidence_map", "Mapping from manuscript claims to artefacts.")
display(claims)

## 16. Limitations, threats to validity, and what would falsify these claims

Stating this explicitly — and exporting it — is part of the contribution. A framework that reports
only its wins is not a scientific artefact.

In [ ]:
# ============================================================
# 16. Limitations (exported for the manuscript)
# ============================================================
LIMITATIONS = f'''
# Limitations and threats to validity
(config_hash={CONFIG_HASH}, generated {ENV["timestamp"]})

1. **Simulation-based validation.** Exact latent ground truth and exact corruption provenance are
   required for the repair-identifiability analysis (C4) and the mechanistic diagnosis (C5); no field
   deployment can supply them. The simulator does not reproduce every electromagnetic, material and
   environmental effect of a physical installation (multipath structure, tag orientation, dense-reader
   interference, metal/liquid attenuation). Results are claims about the *framework*, not about any
   particular warehouse. External validation on physical or openly available RFID deployments is
   identified as the next research stage.

2. **The corruption model is a model.** Faults are injected independently per event apart from reader
   outage. Real RFID failures are temporally bursty and spatially correlated. Correlated-fault
   robustness is not established by these experiments.

3. **Graph availability.** The framework assumes a known process graph G. Where G is unknown,
   partially wrong, or drifting, the graph potentials become a source of bias rather than of
   robustness. Inadmissible transitions are penalised rather than forbidden precisely to bound this
   risk, but graph misspecification is not evaluated here and should be treated as an open question.

4. **Conformal validity.** Split-conformal coverage is guaranteed only under exchangeability. Section 8
   documents that marginal coverage degrades under progressive degradation; the gamma-adaptive
   (Mondrian) variant restores coverage empirically but does not restore the theoretical guarantee
   under arbitrary shift. Reported coverage under shift is empirical, not certified.

5. **Learn-then-Test scope.** The selective-risk guarantee is a statement about the calibration
   distribution. It does not transfer automatically to degraded conditions; under shift the guarantee
   must be re-established on data from the new regime.

6. **Counterfactual minimality is minimality under an assumed cost model.** The edit costs
   (insert/delete/substitute) are set uniformly. A repair that is minimal under uniform costs need not
   be the physically most likely explanation; cost elicitation from deployment data is future work.

7. **Trajectory-level conformal set sizes are beam-enumerated lower bounds.** Exact counting of paths
   above a score threshold is intractable; the reported figures should be read as lower bounds.

8. **Diagnosis label definition.** A_s is defined as *observable route distortion*, not as "some
   corruption fired". This is a deliberate choice to avoid a degenerate label, and alternative
   definitions would change the reported AUROC values.
'''
(BASE_DIR / "Limitations.md").write_text(LIMITATIONS)
print(LIMITATIONS)

In [ ]:
# ============================================================
# 17. Artefact inventory and run summary
# ============================================================
figs = sorted(p.name for p in FIG_DIR.glob("*.png"))
tabs = sorted(p.name for p in TAB_DIR.glob("*.csv"))
texs = sorted(p.name for p in TEX_DIR.glob("*.tex"))
inventory = pd.DataFrame({"Type": ["Figure"]*len(figs) + ["Table"]*len(tabs) + ["LaTeX"]*len(texs),
                          "File": figs + tabs + texs})
inventory.to_csv(BASE_DIR / "Results_Inventory.csv", index=False)

lines = [
    "Beyond the Read - RFID graph-temporal intelligence: run summary",
    "=" * 72,
    f"Project directory : {BASE_DIR}",
    f"Config hash       : {CONFIG_HASH}",
    f"Quick run         : {QUICK_RUN}",
    f"Device            : {DEVICE}",
    f"Assets            : {CFG['n_assets']}   Seeds: {CFG['n_seeds']}",
    "",
    "RECONSTRUCTION", recon.round(4).to_string(index=False), "",
    "CALIBRATION", calib.to_string(index=False), "",
    "CONFORMAL (in-distribution)", conf_tab.to_string(index=False), "",
    "CONFORMAL UNDER SHIFT", shift_df.round(4).to_string(index=False), "",
    "DUAL DIAGNOSIS", diag_tab.to_string(index=False), "",
    "COUNTERFACTUAL IDENTIFIABILITY", ident.to_string(index=False), "",
    "SELECTIVE RISK GUARANTEE", guarantee.to_string(index=False), "",
    "UTILITY", util_summary.to_string(index=False), "",
    "ABLATION", abl.round(4).to_string(index=False), "",
    "PAIRED STATISTICS", stats_df.round(5).to_string(index=False), "",
    "COMPUTATIONAL COST", cost_tab.to_string(index=False), "",
    f"Artefacts: {len(figs)} figures, {len(tabs)} tables, {len(texs)} LaTeX fragments",
]
(BASE_DIR / "Outputs_Summary.txt").write_text("\n".join(lines))
print("\n".join(lines[:12]))
print(f"\nAll artefacts written under: {BASE_DIR}")
display(inventory)

## 18. Protocol for the final manuscript run

1. Set `QUICK_RUN = False` (12k assets, 10 seeds, 40 epochs, 9 stress levels).
2. Runtime → Restart, then Run all, in a clean Colab session with a GPU if available.
3. Confirm the leakage audit in §3.3 prints **PASSED** — if it does not, no number below it is usable.
4. Confirm `Table_12_risk_guarantee.csv` reports `GuaranteeHolds = True`.
5. Copy manuscript numbers **only** from the CSV/LaTeX files in `Tables/` and `LaTeX/`.
6. Archive `config.json`, `Outputs_Summary.txt`, `Limitations.md` and `Results_Inventory.csv`
   alongside the notebook; the config hash printed in every `.tex` header ties each number to its run.
7. For the Data Availability statement: release this notebook plus `config.json`. Every result is
   regenerable from those two files alone — no external dataset is required.

### Suggested headline claims (verify against your own run before use)

* Reconstruction of hidden RFID trajectories in which the process graph enters the **training
  objective** rather than the inference step, yielding markedly higher *exact* route recovery and
  markedly lower residual graph inconsistency than post-hoc graph filtering.
  **Check `Table_22_paired_statistics.csv` before wording this claim.** On edit-similarity the two
  can be statistically tied; the defensible claim is about exact recovery and physical consistency,
  not about every metric. Do not write "dominates" unless the Holm-adjusted test supports it.
* The first RFID counterfactual repair that is **provably minimum-cost** *and* **validated against the
  corruption mechanism that actually occurred**.
* Evidence that **marginal conformal prediction silently loses coverage** as RFID infrastructure
  degrades, and a deployable severity-adaptive correction that restores it.
* A selective-automation layer carrying a **finite-sample risk guarantee** rather than a tuned threshold.